In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet").exists():
    PROJECT_ROOT = Path.cwd().parent

IMAGE_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
PRODUCT_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_products.parquet"

assert IMAGE_DATA_PATH.exists(), f"Image dataset not found: {IMAGE_DATA_PATH}"
assert PRODUCT_DATA_PATH.exists(), f"Product dataset not found: {PRODUCT_DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Image dataset:", IMAGE_DATA_PATH)
print("Product dataset:", PRODUCT_DATA_PATH)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

IMAGE_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)
PRODUCT_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "canonical_products.parquet"
)

assert IMAGE_DATA_PATH.exists(), f"Image dataset not found: {IMAGE_DATA_PATH}"
assert PRODUCT_DATA_PATH.exists(), f"Product dataset not found: {PRODUCT_DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Image dataset found:", IMAGE_DATA_PATH.exists())
print("Product dataset found:", PRODUCT_DATA_PATH.exists())

In [ ]:
images_df = pd.read_parquet(IMAGE_DATA_PATH)
products_df = pd.read_parquet(PRODUCT_DATA_PATH)

print("Image dataset shape:", images_df.shape)
print("Product dataset shape:", products_df.shape)

display(images_df.head())

In [ ]:
print("=== IMAGE DATA QUALITY ===")

print("\nDownload status:")
display(images_df["download_status"].value_counts(dropna=False))

print("\nImage type:")
display(images_df["image_type"].value_counts(dropna=False))

print("\nMissing values:")
display(
    images_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nExact duplicate rows:", images_df.duplicated().sum())

print(
    "\nUnique image IDs:",
    images_df["image_id"].nunique()
)

print(
    "Unique product IDs in image dataset:",
    images_df["canonical_product_id"].nunique()
)

In [ ]:
product_ids = set(products_df["canonical_product_id"].dropna())
image_product_ids = set(images_df["canonical_product_id"].dropna())

orphan_image_products = image_product_ids - product_ids

print("Unique products:", len(product_ids))
print("Unique products represented in image data:", len(image_product_ids))
print("Image product IDs missing from canonical products:",
      len(orphan_image_products))

assert len(orphan_image_products) == 0, (
    "Some image records reference unknown canonical products."
)

In [ ]:
from PIL import Image

print("Pillow is installed and ready.")

In [ ]:
# Step 5: Validate downloaded image files

from pathlib import Path
from PIL import Image

downloaded_df = images_df[
    images_df["download_status"].eq("DOWNLOADED")
].copy()

validation_results = []

for _, row in downloaded_df.iterrows():
    image_path = PROJECT_ROOT / Path(row["local_path"])

    result = {
        "image_id": row["image_id"],
        "canonical_product_id": row["canonical_product_id"],
        "file_exists": image_path.is_file(),
        "readable": False,
        "actual_width": None,
        "actual_height": None,
        "actual_size_bytes": None,
        "error": None,
    }

    try:
        if not image_path.is_file():
            raise FileNotFoundError(f"Image not found: {image_path}")

        with Image.open(image_path) as img:
            img.verify()

        with Image.open(image_path) as img:
            result["actual_width"] = img.width
            result["actual_height"] = img.height

        result["actual_size_bytes"] = image_path.stat().st_size
        result["readable"] = True

    except Exception as exc:
        result["error"] = str(exc)

    validation_results.append(result)

image_validation_df = pd.DataFrame(validation_results)

print("Downloaded records:", len(downloaded_df))
print("Files found:", int(image_validation_df["file_exists"].sum()))
print("Readable images:", int(image_validation_df["readable"].sum()))
print("Unreadable or missing:",
      int((~image_validation_df["readable"]).sum()))

display(image_validation_df.head())

In [ ]:

# Step 6: Compare recorded metadata with actual files

comparison_df = downloaded_df[
    ["image_id", "width", "height", "file_size_bytes"]
].merge(
    image_validation_df[
        [
            "image_id",
            "readable",
            "actual_width",
            "actual_height",
            "actual_size_bytes",
        ]
    ],
    on="image_id",
    how="left",
    validate="one_to_one",
)

# Compare dimensions and sizes only for readable files
readable_df = comparison_df.loc[
    comparison_df["readable"].eq(True)
].copy()

readable_df["dimensions_match"] = (
    readable_df["width"].eq(readable_df["actual_width"])
    & readable_df["height"].eq(readable_df["actual_height"])
)

readable_df["file_size_matches"] = (
    readable_df["file_size_bytes"].eq(
        readable_df["actual_size_bytes"]
    )
)

print("Readable images:", len(readable_df))
print(
    "Dimension mismatches:",
    int((~readable_df["dimensions_match"]).sum()),
)
print(
    "File-size mismatches:",
    int((~readable_df["file_size_matches"]).sum()),
)

display(
    readable_df.loc[
        (~readable_df["dimensions_match"])
        | (~readable_df["file_size_matches"])
    ]
)


In [ ]:

# Step 7: Exact duplicate image analysis

hash_df = images_df[
    images_df["content_hash_sha256"].notna()
    & images_df["content_hash_sha256"].astype("string").str.strip().ne("")
].copy()

repeated_hashes_df = hash_df[
    hash_df.duplicated("content_hash_sha256", keep=False)
].sort_values("content_hash_sha256")

print("Images with SHA-256 hashes:", len(hash_df))
print("Distinct SHA-256 hashes:", hash_df["content_hash_sha256"].nunique())
print("Rows in repeated-hash groups:", len(repeated_hashes_df))
print(
    "Number of repeated-hash groups:",
    repeated_hashes_df["content_hash_sha256"].nunique()
)

display(
    repeated_hashes_df[
        [
            "image_id",
            "canonical_product_id",
            "marketplace",
            "image_type",
            "content_hash_sha256",
        ]
    ]
)


In [ ]:

# Step 8: Classify exact duplicate groups

if repeated_hashes_df.empty:
    print("No repeated SHA-256 hashes found.")
    duplicate_groups_df = pd.DataFrame()
else:
    duplicate_groups_df = (
        repeated_hashes_df.groupby("content_hash_sha256")
        .agg(
            image_count=("image_id", "size"),
            unique_products=("canonical_product_id", "nunique"),
            image_ids=("image_id", list),
            product_ids=("canonical_product_id", list),
        )
        .reset_index()
    )

    duplicate_groups_df["match_type"] = np.where(
        duplicate_groups_df["unique_products"] > 1,
        "CROSS_PRODUCT_MATCH",
        "SAME_PRODUCT_MATCH",
    )

    display(duplicate_groups_df)

    print("\nMatch type summary:")
    display(duplicate_groups_df["match_type"].value_counts())


In [ ]:

# Step 9: Repeated perceptual hashes

phash_df = images_df[
    images_df["perceptual_hash_dhash"].notna()
    & images_df["perceptual_hash_dhash"].astype("string").str.strip().ne("")
].copy()

repeated_phashes_df = phash_df[
    phash_df.duplicated("perceptual_hash_dhash", keep=False)
].sort_values("perceptual_hash_dhash")

print("Images with perceptual hashes:", len(phash_df))
print("Distinct perceptual hashes:",
      phash_df["perceptual_hash_dhash"].nunique())
print("Rows in repeated-hash groups:", len(repeated_phashes_df))
print("Number of repeated-hash groups:",
      repeated_phashes_df["perceptual_hash_dhash"].nunique())

display(
    repeated_phashes_df[
        [
            "image_id",
            "canonical_product_id",
            "image_type",
            "perceptual_hash_dhash",
        ]
    ]
)


In [ ]:

# Step 10: Initial duplicate-analysis summary

summary = {
    "total_image_records": len(images_df),
    "downloaded_records": int(
        images_df["download_status"].eq("DOWNLOADED").sum()
    ),
    "pending_records": int(
        images_df["download_status"].eq("PENDING").sum()
    ),
    "images_with_sha256": int(
        images_df["content_hash_sha256"].notna().sum()
    ),
    "images_with_perceptual_hash": int(
        images_df["perceptual_hash_dhash"].notna().sum()
    ),
    "exact_duplicate_hash_groups": (
        0 if duplicate_groups_df.empty
        else len(duplicate_groups_df)
    ),
    "repeated_perceptual_hash_groups": (
        repeated_phashes_df["perceptual_hash_dhash"].nunique()
    ),
}

display(pd.Series(summary, name="value").to_frame())


In [ ]:

# Step 11: Inspect exact duplicate candidates in detail

if repeated_hashes_df.empty:
    print("No exact duplicate candidates found.")
else:
    for hash_value, group in repeated_hashes_df.groupby(
        "content_hash_sha256"
    ):
        print("\n" + "=" * 60)
        print("SHA-256:", hash_value)
        print("Number of image records:", len(group))
        print(
            "Unique products:",
            group["canonical_product_id"].nunique()
        )

        display(
            group[
                [
                    "image_id",
                    "canonical_product_id",
                    "marketplace",
                    "image_type",
                    "local_path",
                ]
            ]
        )


In [ ]:

# Step 12: Inspect repeated perceptual hashes

if repeated_phashes_df.empty:
    print("No repeated perceptual-hash candidates found.")
else:
    for hash_value, group in repeated_phashes_df.groupby(
        "perceptual_hash_dhash"
    ):
        print("\n" + "=" * 60)
        print("Perceptual hash:", hash_value)
        print("Number of image records:", len(group))
        print(
            "Unique products:",
            group["canonical_product_id"].nunique()
        )

        display(
            group[
                [
                    "image_id",
                    "canonical_product_id",
                    "image_type",
                    "local_path",
                ]
            ]
        )


In [ ]:

# Step 13: Prepare perceptual hashes for Hamming-distance analysis

def hamming_distance(hash_a, hash_b):
    """Count differing bits between two hexadecimal hashes."""
    a = int(str(hash_a), 16)
    b = int(str(hash_b), 16)
    return (a ^ b).bit_count()


phash_df = images_df[
    images_df["perceptual_hash_dhash"].notna()
    & images_df["perceptual_hash_dhash"].astype("string").str.strip().ne("")
].copy()

print("Images with perceptual hashes:", len(phash_df))
print("Unique perceptual hashes:", phash_df["perceptual_hash_dhash"].nunique())

# Example to verify the function
example_hash = phash_df["perceptual_hash_dhash"].iloc[0]
print("Example hash:", example_hash)
print("Distance from itself:", hamming_distance(example_hash, example_hash))


In [ ]:

# Step 14: Find near-duplicate candidates among available hashes

records = phash_df[
    ["image_id", "canonical_product_id", "perceptual_hash_dhash"]
].drop_duplicates("image_id").to_dict("records")

near_duplicate_pairs = []

for i in range(len(records)):
    for j in range(i + 1, len(records)):
        a = records[i]
        b = records[j]

        if a["perceptual_hash_dhash"] == b["perceptual_hash_dhash"]:
            continue

        distance = hamming_distance(
            a["perceptual_hash_dhash"],
            b["perceptual_hash_dhash"],
        )

        if distance <= 2:
            near_duplicate_pairs.append({
                "image_id_1": a["image_id"],
                "product_id_1": a["canonical_product_id"],
                "image_id_2": b["image_id"],
                "product_id_2": b["canonical_product_id"],
                "hamming_distance": distance,
                "same_product": (
                    a["canonical_product_id"] == b["canonical_product_id"]
                ),
            })

near_duplicates_df = pd.DataFrame(near_duplicate_pairs)

print("Near-duplicate pairs found:", len(near_duplicates_df))

if near_duplicates_df.empty:
    print("No distinct-hash pairs found at distance <= 2.")
else:
    display(
        near_duplicates_df.sort_values("hamming_distance")
    )


In [ ]:

# Step 16: Final image-validation report

total_images = len(images_df)

downloaded_count = int(
    images_df["download_status"].eq("DOWNLOADED").sum()
)
pending_count = int(
    images_df["download_status"].eq("PENDING").sum()
)

sha256_count = int(
    images_df["content_hash_sha256"].notna().sum()
)
phash_count = int(
    images_df["perceptual_hash_dhash"].notna().sum()
)

report = pd.DataFrame({
    "Metric": [
        "Total image records",
        "Downloaded records",
        "Pending records",
        "Images with SHA-256 hashes",
        "Images with perceptual hashes",
        "Exact duplicate hash groups",
        "Repeated perceptual hash groups",
        "Near-duplicate pairs (Hamming distance <= 2)",
    ],
    "Value": [
        total_images,
        downloaded_count,
        pending_count,
        sha256_count,
        phash_count,
        len(duplicate_groups_df),
        repeated_phashes_df["perceptual_hash_dhash"].nunique(),
        len(near_duplicates_df),
    ],
})

display(report)


## Computer Vision — Initial Image Validation

### Objective
Establish the quality, availability, and duplicate-image baseline for the marketplace image dataset before implementing more advanced computer-vision features.

### Current dataset
The canonical image dataset contains 3,909 image records. Of these, 520 are marked as downloaded and 3,389 remain pending. The available downloaded records have image fingerprints that can be used for initial duplicate analysis.

### Validation performed
- Inspected image metadata, image types, and download statuses.
- Checked image-to-product mappings against canonical product records.
- Validated local image readability and compared recorded metadata with actual files.
- Investigated repeated SHA-256 and perceptual-hash values.
- Prepared perceptual-hash Hamming-distance analysis for candidate near-duplicate pairs.

### Limitations
The current fingerprint-based analysis covers only records with available hashes. Pending image records are not yet represented in this analysis. Repeated hashes and near-duplicate candidates are not, by themselves, evidence of fraudulent behavior.

### Next steps
1. Review and document the actual duplicate-analysis results.
2. Decide whether to expand image downloads to improve coverage.
3. Develop and validate duplicate-image detection.
4. Investigate OCR and image-text consistency after image coverage and quality are sufficiently established.

In [ ]:
# Step 23: Inspect image coverage by status and image type

print("=== STATUS BY IMAGE TYPE ===")
display(pd.crosstab(
    images_df["image_type"],
    images_df["download_status"],
    margins=True
))

print("\n=== SOURCE PROVENANCE ===")
display(images_df["source_provenance"].value_counts(dropna=False))

print("\n=== PRODUCTS WITH IMAGES ===")
print(
    "Unique image records:",
    images_df["image_id"].nunique()
)
print(
    "Unique products represented:",
    images_df["canonical_product_id"].nunique()
)
print(
    "Unique products with at least one downloaded image:",
    images_df.loc[
        images_df["download_status"].eq("DOWNLOADED"),
        "canonical_product_id"
    ].nunique()
)

In [ ]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nCanonical dataset exists:")
print(Path("data/processed/canonical_images.parquet").exists())

print("\nInterim manifest exists:")
print(Path("data/interim/image_manifest.parquet").exists())

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer"
)

canonical_path = PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
interim_path = PROJECT_ROOT / "data" / "interim" / "image_manifest.parquet"

print("Canonical exists:", canonical_path.exists())
print("Interim exists:", interim_path.exists())

In [ ]:
canonical_df = pd.read_parquet(canonical_path)
interim_df = pd.read_parquet(interim_path)

print("Canonical dataset:", canonical_df.shape)
print("Interim manifest:", interim_df.shape)

In [ ]:
print("=== DATASET SIZES ===")
print("Canonical images:", canonical_df.shape)
print("Interim manifest:", interim_df.shape)

print("\n=== INTERIM COLUMNS ===")
print(interim_df.columns.tolist())

print("\n=== DOWNLOAD STATUS ===")
if "download_status" in interim_df.columns:
    display(interim_df["download_status"].value_counts(dropna=False))

print("\n=== IMAGE ID OVERLAP ===")
if "image_id" in canonical_df.columns and "image_id" in interim_df.columns:
    canonical_ids = set(canonical_df["image_id"].dropna())
    interim_ids = set(interim_df["image_id"].dropna())

    print("Canonical unique IDs:", len(canonical_ids))
    print("Interim unique IDs:", len(interim_ids))
    print("Shared IDs:", len(canonical_ids & interim_ids))
    print("Only in interim:", len(interim_ids - canonical_ids))
    print("Only in canonical:", len(canonical_ids - interim_ids))

print("\n=== IMAGE TYPES ===")
if "image_type" in interim_df.columns:
    display(interim_df["image_type"].value_counts(dropna=False))

In [ ]:
print("=== CANONICAL COLUMNS ===")
print(canonical_df.columns.tolist())

print("\n=== CANONICAL MISSING VALUES ===")
display(canonical_df.isna().sum().sort_values(ascending=False))

print("\n=== INTERIM MISSING VALUES ===")
display(interim_df.isna().sum().sort_values(ascending=False))

print("\n=== MATCH VALIDATION ===")
matched = canonical_df.merge(
    interim_df[[
        "image_id",
        "canonical_product_id",
        "image_url",
        "content_hash_sha256",
        "perceptual_hash_dhash",
        "ocr_text",
        "download_status"
    ]],
    on="image_id",
    how="inner",
    suffixes=("_canonical", "_interim"),
    validate="one_to_one"
)

print("Matched records:", len(matched))
print(
    "Product ID mismatches:",
    (
        matched["canonical_product_id_canonical"]
        != matched["canonical_product_id_interim"]
    ).sum()
)

print(
    "Image URL mismatches:",
    (
        matched["image_url_canonical"].fillna("")
        != matched["image_url_interim"].fillna("")
    ).sum()
)

print("\n=== EXISTING CANONICAL IMAGE ANALYSIS COVERAGE ===")
for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score"
]:
    if col in canonical_df.columns:
        print(f"{col}: {canonical_df[col].notna().sum()} non-missing values")

In [ ]:
print("=== MATCH VALIDATION ===")

matched = canonical_df.merge(
    interim_df[
        [
            "image_id",
            "canonical_product_id",
            "image_url",
            "content_hash_sha256",
            "perceptual_hash_dhash",
            "ocr_text",
            "download_status",
        ]
    ],
    on="image_id",
    how="inner",
    suffixes=("_canonical", "_interim"),
    validate="one_to_one",
)

print("Matched records:", len(matched))

print(
    "Product ID mismatches:",
    (
        matched["canonical_product_id_canonical"]
        != matched["canonical_product_id_interim"]
    ).sum(),
)

print(
    "Image URL mismatches:",
    (
        matched["image_url_canonical"].fillna("")
        != matched["image_url_interim"].fillna("")
    ).sum(),
)

print("\n=== CANONICAL ANALYSIS COVERAGE ===")

for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
]:
    if col in canonical_df.columns:
        print(
            f"{col}: "
            f"{canonical_df[col].notna().sum()} / {len(canonical_df)} "
            "non-missing"
        )

print("\n=== INTERIM ANALYSIS COVERAGE ===")

for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
]:
    if col in interim_df.columns:
        print(
            f"{col}: "
            f"{interim_df[col].notna().sum()} / {len(interim_df)} "
            "non-missing"
        )

In [ ]:
print("=== OCR COVERAGE ===")

ocr_available = interim_df["ocr_text"].notna() & (
    interim_df["ocr_text"].astype("string").str.strip() != ""
)

print("Total downloaded images:", len(interim_df))
print("Images with usable OCR text:", int(ocr_available.sum()))
print("Images without usable OCR text:", int((~ocr_available).sum()))
print("OCR coverage:", f"{ocr_available.mean() * 100:.2f}%")

print("\n=== PACKAGING CONSISTENCY SCORES ===")

display(
    interim_df["packaging_consistency_score"]
    .describe()
    .to_frame("score")
)

print("\n=== SCORES BY OCR AVAILABILITY ===")

interim_df = interim_df.copy()
interim_df["ocr_available"] = ocr_available

display(
    interim_df.groupby("ocr_available")[
        "packaging_consistency_score"
    ].agg(["count", "mean", "median", "min", "max"])
)

print("\n=== OCR EXAMPLES ===")

display(
    interim_df.loc[
        ocr_available,
        ["image_id", "title", "ocr_text", "packaging_consistency_score"]
    ].head(10)
)

In [ ]:
print("=== PACKAGING SCORE QUALITY CHECK ===")

scores = pd.to_numeric(
    interim_df["packaging_consistency_score"],
    errors="coerce"
)

print("Missing scores:", scores.isna().sum())
print("Scores below 0:", (scores < 0).sum())
print("Scores above 1:", (scores > 1).sum())

print("\n=== SCORE DISTRIBUTION ===")
display(scores.value_counts(dropna=False).sort_index())

print("\n=== OCR AVAILABILITY VS SCORE ===")
ocr_available = (
    interim_df["ocr_text"].fillna("").astype(str).str.strip().ne("")
)

display(
    interim_df.assign(
        ocr_available=ocr_available,
        packaging_score=scores
    )
    .groupby("ocr_available")["packaging_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

print("\n=== CHECK NEUTRAL SCORES WITHOUT OCR ===")
no_ocr = interim_df.loc[~ocr_available].copy()

print("Images without OCR:", len(no_ocr))
print(
    "Images without OCR assigned score 0.50:",
    int(
        pd.to_numeric(
            no_ocr["packaging_consistency_score"],
            errors="coerce"
        ).eq(0.50).sum()
    )
)

In [ ]:
print("=== NEUTRAL SCORE INVESTIGATION ===")

scores = pd.to_numeric(
    interim_df["packaging_consistency_score"],
    errors="coerce"
)

ocr_available = (
    interim_df["ocr_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

neutral_score = scores.eq(0.50)

print("Total images:", len(interim_df))
print("Images scored 0.50:", int(neutral_score.sum()))

print(
    "Score 0.50 WITH OCR:",
    int((neutral_score & ocr_available).sum())
)

print(
    "Score 0.50 WITHOUT OCR:",
    int((neutral_score & ~ocr_available).sum())
)

print(
    "Non-neutral scores WITHOUT OCR:",
    int((~neutral_score & ~ocr_available).sum())
)

print("\n=== SCORE AND OCR CROSS-TABULATION ===")

display(
    pd.crosstab(
        ocr_available,
        neutral_score,
        rownames=["OCR available"],
        colnames=["Score equals 0.50"],
        margins=True
    )
)

print("\n=== EXAMPLES OF NEUTRAL SCORES WITH OCR ===")

display(
    interim_df.loc[
        neutral_score & ocr_available,
        [
            "image_id",
            "title",
            "ocr_text",
            "packaging_consistency_score"
        ]
    ].head(10)
)

In [ ]:
print("=== EXACT DUPLICATE SUMMARY ===")

exact_hashes = (
    canonical_df.loc[
        canonical_df["content_hash_sha256"].notna()
        & canonical_df["content_hash_sha256"].ne(""),
        ["image_id", "canonical_product_id", "content_hash_sha256"]
    ]
)

exact_groups = exact_hashes[
    exact_hashes.duplicated("content_hash_sha256", keep=False)
].sort_values("content_hash_sha256")

print("Records with repeated exact hashes:", len(exact_groups))
print(
    "Distinct repeated exact-hash groups:",
    exact_groups["content_hash_sha256"].nunique()
)

display(exact_groups.head(20))

print("\n=== PERCEPTUAL DUPLICATE SUMMARY ===")

perceptual_hashes = (
    canonical_df.loc[
        canonical_df["perceptual_hash_dhash"].notna()
        & canonical_df["perceptual_hash_dhash"].ne(""),
        ["image_id", "canonical_product_id", "perceptual_hash_dhash"]
    ]
)

perceptual_groups = perceptual_hashes[
    perceptual_hashes.duplicated("perceptual_hash_dhash", keep=False)
].sort_values("perceptual_hash_dhash")

print("Records with repeated perceptual hashes:", len(perceptual_groups))
print(
    "Distinct repeated perceptual-hash groups:",
    perceptual_groups["perceptual_hash_dhash"].nunique()
)

display(perceptual_groups.head(20))

In [ ]:
print("=== INVESTIGATE EXACT DUPLICATE GROUP ===")

duplicate_details = exact_groups.merge(
    interim_df[
        [
            "image_id",
            "title",
            "brand",
            "category",
            "image_url",
            "local_path",
            "source_provenance",
        ]
    ],
    on="image_id",
    how="left",
    validate="one_to_one",
)

display(
    duplicate_details[
        [
            "image_id",
            "canonical_product_id",
            "title",
            "brand",
            "category",
            "image_url",
            "local_path",
            "source_provenance",
        ]
    ].reset_index(drop=True)
)

print("\n=== UNIQUE SOURCE URLS ===")
print("Unique image URLs:", duplicate_details["image_url"].nunique())
print("\n=== UNIQUE PRODUCT TITLES ===")
print("Unique titles:", duplicate_details["title"].nunique())

In [ ]:
print("=== SOURCE URL DUPLICATE INVESTIGATION ===")

url_col = "image_url"

url_counts = (
    interim_df.dropna(subset=[url_col])
    .groupby(url_col)
    .agg(
        image_count=("image_id", "nunique"),
        product_count=("canonical_product_id", "nunique"),
        titles=("title", lambda s: list(s.dropna().unique())),
        categories=("category", lambda s: list(s.dropna().unique())),
        brands=("brand", lambda s: list(s.dropna().unique())),
    )
    .sort_values("product_count", ascending=False)
)

display(url_counts[url_counts["product_count"] > 1].head(10))

print("\n=== EXACT HASH GROUP CHECK ===")
print("Repeated exact-hash records:", len(exact_groups))
print("Distinct product IDs:", exact_groups["canonical_product_id"].nunique())
print("Distinct source URLs:", duplicate_details["image_url"].nunique())

In [ ]:
print("=== REUSED IMAGE URLS ===")

url_analysis = (
    interim_df.dropna(subset=["image_url"])
    .groupby("image_url")
    .agg(
        image_count=("image_id", "nunique"),
        product_count=("canonical_product_id", "nunique"),
        titles=("title", lambda s: list(s.dropna().unique())),
        categories=("category", lambda s: list(s.dropna().unique())),
        brands=("brand", lambda s: list(s.dropna().unique())),
    )
    .reset_index()
)

reused_urls = (
    url_analysis[url_analysis["product_count"] > 1]
    .sort_values("product_count", ascending=False)
)

print("Reused URL groups:", len(reused_urls))
display(reused_urls.head(10))

In [ ]:
print("=== SECOND PERCEPTUAL-HASH GROUP ===")

second_hash = "0c4d4d4d4d4d4d4d"

second_group = canonical_df.loc[
    canonical_df["perceptual_hash_dhash"].eq(second_hash)
].copy()

display(
    second_group[
        [
            "image_id",
            "canonical_product_id",
            "marketplace",
            "image_url",
            "local_path",
            "content_hash_sha256",
            "perceptual_hash_dhash",
        ]
    ]
)

print("Records:", len(second_group))
print(
    "Unique exact hashes:",
    second_group["content_hash_sha256"].nunique()
)

In [ ]:
print("=== NEAR-DUPLICATE ANALYSIS ===")

print("Candidate pairs:", len(near_duplicate_pairs))

if near_duplicate_pairs:
    display(near_duplicate_pairs[:20])
else:
    print("No near-duplicate candidate pairs found.")

## Computer Vision — Preliminary Findings

### Dataset coverage
- Canonical image manifest: 3,909 records.
- Downloaded primary images: 520.
- All 520 downloaded image IDs match existing canonical records.
- Product ID and image URL mismatches between the canonical dataset and interim manifest: zero.

### Duplicate-image investigation
- Five records share one exact SHA-256 hash.
- The five records have different product IDs and titles, span multiple product categories, and share the same source image URL.
- Two perceptual-hash groups were identified, containing seven records in total.
- Repeated perceptual hashes indicate candidate visual matches, not confirmed fraud.

### Packaging consistency
- All 520 downloaded images have a numeric packaging-consistency score within the expected 0–1 range.
- A substantial number of images receive a neutral score of 0.50.
- Neutral scores must be interpreted in the context of OCR availability and the scoring implementation.

### Limitations and next actions
- The analysis currently covers 520 downloaded primary images, not all 3,909 image records.
- Investigate reused image URLs and validate candidate near-duplicates.
- Review OCR coverage and packaging-score semantics before using these features in the risk engine.
- Keep image-source data-quality anomalies separate from confirmed fraud labels.

In [ ]:
near_duplicates_df = pd.DataFrame(near_duplicate_pairs)

print("=== NEAR-DUPLICATE CANDIDATES ===")
display(
    near_duplicates_df.sort_values(
        "hamming_distance"
    ).reset_index(drop=True)
)

print("\n=== DISTANCE DISTRIBUTION ===")
display(
    near_duplicates_df["hamming_distance"]
    .value_counts()
    .sort_index()
    .rename_axis("hamming_distance")
    .to_frame("candidate_pairs")
)

print("\n=== SAME-PRODUCT CHECK ===")
print(
    "Pairs from different products:",
    int((~near_duplicates_df["same_product"]).sum())
)

In [ ]:
print("=== NEAR-DUPLICATE IMAGE FILES ===")

image_lookup = (
    canonical_df[
        ["image_id", "canonical_product_id", "local_path"]
    ]
    .drop_duplicates("image_id")
    .set_index("image_id")
)

for _, pair in near_duplicates_df.iterrows():
    print(
        f"\nPair: {pair['image_id_1']} <-> {pair['image_id_2']}"
    )

    for image_id in [pair["image_id_1"], pair["image_id_2"]]:
        if image_id in image_lookup.index:
            path = image_lookup.loc[image_id, "local_path"]
            print(f"  {image_id}: {path}")
        else:
            print(f"  {image_id}: not found in canonical image records")

## Final Computer Vision Analysis — Current Findings

### 1. Dataset coverage
- Canonical image records: 3,909.
- Downloaded primary images: 520.
- Products represented by downloaded images: 520, subject to verification of unique product counts.
- All downloaded image IDs match canonical records.
- No product ID or image URL mismatches were found between the canonical dataset and interim manifest.

### 2. Exact duplicate detection
- Five records share one repeated SHA-256 hash.
- The five records have different product IDs and titles.
- They share one image URL across multiple product categories.
- This is a potential image-source or dataset-mapping anomaly that requires investigation, not a confirmed fraud finding.

### 3. Perceptual similarity
- Two repeated perceptual-hash groups contain seven records in total.
- Five near-duplicate candidate pairs were identified.
- Candidate Hamming distances range from 1 to 2.
- All five candidate pairs involve different product IDs.
- Visual inspection is needed to validate these candidate matches.

### 4. OCR and packaging consistency
- OCR text is available for 204 of the 520 downloaded images.
- All packaging-consistency scores are within the expected 0–1 range.
- A substantial number of records receive the neutral score 0.50.
- The score's meaning must be validated against OCR availability and the scoring implementation.

### 5. Limitations
- Current image-analysis coverage is limited to the 520 downloaded primary images.
- The remaining canonical image records have not been validated through the same downloaded-image workflow.
- Hash similarity alone cannot establish product fraud or seller intent.
- Packaging-consistency scores should not be treated as calibrated fraud probabilities.

### 6. Next actions
- Visually inspect duplicate candidates.
- Investigate reused source image URLs.
- Validate the packaging-consistency implementation.
- Integrate validated image signals into the multimodal risk engine with explicit evidence and coverage indicators.

In [ ]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 1. Set the project root
PROJECT_ROOT = Path(
    r"C:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer"
)

IMAGE_MANIFEST = (
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)

# 2. Load the existing image manifest
if not IMAGE_MANIFEST.exists():
    raise FileNotFoundError(
        f"Image manifest not found:\n{IMAGE_MANIFEST}\n"
        f"Check that the project folder exists."
    )

df = pd.read_parquet(IMAGE_MANIFEST)

print("Manifest shape:", df.shape)

# 3. Select downloaded images with perceptual hashes
df = df[
    df["download_status"].eq("DOWNLOADED")
    & df["perceptual_hash_dhash"].notna()
].drop_duplicates("image_id").reset_index(drop=True)

print("Images available for comparison:", len(df))


# 4. Calculate Hamming distance between dHash values
def hamming_distance(hash_a, hash_b):
    return (
        int(str(hash_a), 16) ^ int(str(hash_b), 16)
    ).bit_count()


# 5. Find near-duplicate candidates
candidates = []

for i, j in combinations(range(len(df)), 2):
    a = df.iloc[i]
    b = df.iloc[j]

    # Skip pairs with identical file contents
    if (
        pd.notna(a["content_hash_sha256"])
        and a["content_hash_sha256"] == b["content_hash_sha256"]
    ):
        continue

    distance = hamming_distance(
        a["perceptual_hash_dhash"],
        b["perceptual_hash_dhash"]
    )

    if distance <= 3:
        candidates.append((distance, i, j))

candidates.sort(key=lambda item: item[0])

print("Near-duplicate candidates:", len(candidates))


# 6. Display up to six pairs for manual inspection
shown = 0

for distance, i, j in candidates:
    a = df.iloc[i]
    b = df.iloc[j]

    path_a = PROJECT_ROOT / str(a["local_path"])
    path_b = PROJECT_ROOT / str(b["local_path"])

    # Validate paths before opening
    if not path_a.is_file() or not path_b.is_file():
        print("Skipping pair: local image file missing.")
        continue

    try:
        with Image.open(path_a) as img:
            image_a = img.convert("RGB").copy()

        with Image.open(path_b) as img:
            image_b = img.convert("RGB").copy()

    except Exception as exc:
        print(f"Skipping unreadable pair: {exc}")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(image_a)
    axes[0].set_title(str(a["canonical_product_id"]))
    axes[0].axis("off")

    axes[1].imshow(image_b)
    axes[1].set_title(str(b["canonical_product_id"]))
    axes[1].axis("off")

    fig.suptitle(
        f"dHash distance: {distance} | "
        f"Same URL: {a['image_url'] == b['image_url']}"
    )

    plt.tight_layout()
    plt.show()

    shown += 1

    if shown >= 6:
        break

print(f"\nDisplayed {shown} image pairs for manual inspection.")

In [ ]:
print("Near-duplicate candidates:", len(candidates))
print("Image pairs displayed:", shown)

In [ ]:
# Display all candidate pairs with their product IDs and distances

for rank, (distance, i, j) in enumerate(candidates, start=1):
    a = df.iloc[i]
    b = df.iloc[j]

    print(f"\nCandidate {rank} | dHash distance: {distance}")
    print("Image A:", a["image_id"])
    print("Product A:", a["canonical_product_id"])
    print("Image B:", b["image_id"])
    print("Product B:", b["canonical_product_id"])
    print("Same URL:", a["image_url"] == b["image_url"])
    print("Same SHA-256:", a["content_hash_sha256"] == b["content_hash_sha256"])

In [ ]:
candidate_rows = []

for distance, i, j in candidates:
    a = df.iloc[i]
    b = df.iloc[j]

    candidate_rows.append({
        "distance": distance,
        "product_a": a["canonical_product_id"],
        "product_b": b["canonical_product_id"],
        "same_url": a["image_url"] == b["image_url"],
        "same_sha256": (
            a["content_hash_sha256"] == b["content_hash_sha256"]
        ),
    })

candidate_table = pd.DataFrame(candidate_rows)

display(candidate_table)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

# Collect distinct image indices from all candidate pairs
candidate_indices = sorted({
    idx
    for _, i, j in candidates
    for idx in (i, j)
})

print("Distinct images in candidate pairs:", len(candidate_indices))

# Display images in batches of 8
for start in range(0, len(candidate_indices), 8):
    batch = candidate_indices[start:start + 8]

    fig, axes = plt.subplots(
        2, 4, figsize=(14, 8)
    )
    axes = axes.flatten()

    for ax, idx in zip(axes, batch):
        row = df.iloc[idx]
        path = PROJECT_ROOT / str(row["local_path"])

        try:
            with Image.open(path) as img:
                ax.imshow(img.convert("RGB"))

            ax.set_title(
                f"{row['canonical_product_id']}\n"
                f"dHash: {row['perceptual_hash_dhash']}",
                fontsize=8
            )
        except Exception as exc:
            ax.set_title(f"Could not load: {exc}", fontsize=7)

        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for rank, (distance, i, j) in enumerate(candidates[:11], start=1):
    a = df.iloc[i]
    b = df.iloc[j]

    path_a = PROJECT_ROOT / str(a["local_path"])
    path_b = PROJECT_ROOT / str(b["local_path"])

    with Image.open(path_a) as img:
        image_a = img.convert("RGB").copy()

    with Image.open(path_b) as img:
        image_b = img.convert("RGB").copy()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].imshow(image_a)
    axes[0].set_title(str(a["canonical_product_id"]))
    axes[0].axis("off")

    axes[1].imshow(image_b)
    axes[1].set_title(str(b["canonical_product_id"]))
    axes[1].axis("off")

    fig.suptitle(
        f"Pair {rank} | dHash distance = {distance}"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
from IPython.display import display

annotations = pd.DataFrame([
    {
        "pair_number": 7,
        "distance": 3,
        "image_a": "amazon_B00XYMG1HC",
        "image_b": "amazon_B07QMZS32S",
        "visual_label": "different_products",
    },
    {
        "pair_number": 8,
        "distance": 3,
        "image_a": "amazon_B00XYMG1HC",
        "image_b": "amazon_B0BWZX4TQ1",
        "visual_label": "different_products",
    },
    {
        "pair_number": 9,
        "distance": 3,
        "image_a": "amazon_B0B6DPNJHQ",
        "image_b": "amazon_B00AQ25NB8",
        "visual_label": "different_products",
    },
])

display(annotations)

In [ ]:
import pandas as pd
from IPython.display import display

annotation_rows = []

for pair_number, (distance, i, j) in enumerate(candidates, start=1):
    a = df.iloc[i]
    b = df.iloc[j]

    annotation_rows.append({
        "pair_number": pair_number,
        "distance": distance,
        "image_a": a["canonical_product_id"],
        "image_b": b["canonical_product_id"],
        "visual_label": "pending_review",
    })

annotations = pd.DataFrame(annotation_rows)

# Record the three labels we have already inspected
known_labels = {
    7: "different_products",
    8: "different_products",
    9: "different_products",
}

annotations["visual_label"] = annotations["pair_number"].map(
    known_labels
).fillna("pending_review")

display(annotations)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

reviewed_labels = {
    7: "different_products",
    8: "different_products",
    9: "different_products",
}

for pair_number, (distance, i, j) in enumerate(candidates, start=1):
    if pair_number in reviewed_labels:
        continue

    a = df.iloc[i]
    b = df.iloc[j]

    path_a = PROJECT_ROOT / str(a["local_path"])
    path_b = PROJECT_ROOT / str(b["local_path"])

    with Image.open(path_a) as img:
        image_a = img.convert("RGB").copy()

    with Image.open(path_b) as img:
        image_b = img.convert("RGB").copy()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].imshow(image_a)
    axes[0].set_title(a["canonical_product_id"])
    axes[0].axis("off")

    axes[1].imshow(image_b)
    axes[1].set_title(b["canonical_product_id"])
    axes[1].axis("off")

    fig.suptitle(
        f"Pair {pair_number} | dHash distance = {distance}"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
try:
    import torch
    import torchvision

    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())

except ImportError as e:
    print("Embedding dependencies are not available:", e)

In [ ]:
import torch
import torchvision
from torchvision import models

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Remove the final classification layer to obtain image features
model.fc = torch.nn.Identity()
model.eval()

print("Model loaded successfully!")
print("Device: CPU")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from PIL import Image
from pathlib import Path
from torchvision import models

# Project paths
PROJECT_ROOT = Path(
    r"C:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer"
)
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"

# Load manifest
image_df = pd.read_parquet(MANIFEST_PATH)

image_df = (
    image_df[
        image_df["download_status"].eq("DOWNLOADED")
        & image_df["local_path"].notna()
    ]
    .drop_duplicates("image_id")
    .reset_index(drop=True)
)

# Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = torch.nn.Identity()
model.eval()

preprocess = weights.transforms()

embeddings = []
valid_rows = []
failed_images = []

# Extract embeddings on CPU
with torch.inference_mode():
    for idx, row in image_df.iterrows():
        image_path = PROJECT_ROOT / str(row["local_path"])

        try:
            with Image.open(image_path) as img:
                image = img.convert("RGB")
                tensor = preprocess(image).unsqueeze(0)

            vector = model(tensor)
            vector = F.normalize(vector, p=2, dim=1)

            embeddings.append(vector.squeeze(0).numpy())
            valid_rows.append(idx)

        except Exception as exc:
            failed_images.append({
                "image_id": row["image_id"],
                "error": str(exc),
            })

        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(image_df)} images")

embedding_df = image_df.iloc[valid_rows].reset_index(drop=True)
embedding_matrix = np.stack(embeddings) if embeddings else np.empty((0, 512))

print("\n--- EMBEDDING RESULTS ---")
print("Images processed:", len(embedding_df))
print("Failed images:", len(failed_images))
print("Embedding matrix shape:", embedding_matrix.shape)

In [ ]:
import numpy as np
import pandas as pd

from itertools import combinations

# Each row is an L2-normalized 512-dimensional embedding.
# Dot product therefore equals cosine similarity.
similarity_matrix = embedding_matrix @ embedding_matrix.T

pairs = []

for i, j in combinations(range(len(embedding_df)), 2):
    a = embedding_df.iloc[i]
    b = embedding_df.iloc[j]

    d_hash_distance = (
        int(a["perceptual_hash_dhash"], 16)
        ^ int(b["perceptual_hash_dhash"], 16)
    ).bit_count()

    same_sha256 = (
        pd.notna(a["content_hash_sha256"])
        and a["content_hash_sha256"] == b["content_hash_sha256"]
    )

    pairs.append({
        "image_a": a["image_id"],
        "image_b": b["image_id"],
        "product_a": a["canonical_product_id"],
        "product_b": b["canonical_product_id"],
        "dhash_distance": d_hash_distance,
        "cosine_similarity": float(similarity_matrix[i, j]),
        "same_sha256": same_sha256,
    })

pair_results = pd.DataFrame(pairs)

print("Pairs compared:", len(pair_results))

print("\n--- TOP 20 RESNET SIMILARITY PAIRS ---")
display(
    pair_results.sort_values(
        "cosine_similarity", ascending=False
    ).head(20).round({"cosine_similarity": 4})
)

print("\n--- COSINE SIMILARITY DISTRIBUTION ---")
print(
    pair_results["cosine_similarity"]
    .describe()
    .round(4)
    .to_string()
)

In [ ]:
non_exact = pair_results[~pair_results["same_sha256"]].copy()

non_exact = non_exact.sort_values(
    "cosine_similarity",
    ascending=False
).reset_index(drop=True)

display(
    non_exact[
        [
            "product_a",
            "product_b",
            "dhash_distance",
            "cosine_similarity",
            "same_sha256"
        ]
    ].head(20).round({"cosine_similarity": 4})
)

In [ ]:
print("Image A path:", path_a)
print("Image A exists:", Path(path_a).exists())

print("Image B path:", path_b)
print("Image B exists:", Path(path_b).exists())

In [ ]:
print(embedding_df[["image_id", "local_path"]].head(10))
print("\nMissing paths:", embedding_df["local_path"].isna().sum())

In [ ]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nDoes the image cache exist here?")
print(Path("data/raw/images/cache").resolve().exists())

In [ ]:
cwd = Path.cwd()

for folder in [
    cwd / "data/raw/images/cache",
    cwd.parent / "data/raw/images/cache",
    cwd.parent.parent / "data/raw/images/cache",
]:
    print(folder.resolve(), "->", folder.exists())

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent

print("Project root:", project_root)
print("Project root exists:", project_root.exists())

for folder in ["data", "data/raw", "data/processed", "models"]:
    p = project_root / folder
    print(f"{folder}: exists={p.exists()}")

In [ ]:
target = "B00XYMG1HC_img_0.jpg"

matches = list(project_root.rglob(target))

print("Matches found:", len(matches))

for match in matches[:10]:
    print(match)

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent

# Create a lookup from image ID to the correct absolute file path
image_path_lookup = {
    path.stem: path
    for path in (project_root / "data/raw/images/cache").glob("*")
    if path.is_file()
}

print("Images found in cache:", len(image_path_lookup))
print("Example:", next(iter(image_path_lookup.items()), "No images found"))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

top_pairs = non_exact.head(6)

for rank, (_, row) in enumerate(top_pairs.iterrows(), start=1):
    image_a = image_path_lookup.get(row["image_a"])
    image_b = image_path_lookup.get(row["image_b"])

    if image_a is None or image_b is None:
        print(
            f"Pair {rank}: image not found in cache.",
            row["image_a"], row["image_b"]
        )
        continue

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(Image.open(image_a).convert("RGB"))
    axes[0].set_title(row["product_a"], fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(Image.open(image_b).convert("RGB"))
    axes[1].set_title(row["product_b"], fontsize=9)
    axes[1].axis("off")

    fig.suptitle(
        f"Pair {rank} | Cosine: {row['cosine_similarity']:.4f}"
        f" | dHash distance: {row['dhash_distance']}",
        fontsize=11
    )

    plt.tight_layout()
    plt.show()

In [ ]:
print("Sample image IDs:")
print(embedding_df["image_id"].head(5).tolist())

print("\nSample cache filenames:")
for path in list((project_root / "data/raw/images/cache").glob("*"))[:5]:
    print(path.name)

In [ ]:
image_path_lookup = {
    path.stem: path
    for path in (project_root / "data/raw/images/cache").glob("*")
    if path.is_file()
}

test_id = non_exact.iloc[0]["image_a"]

print("Image ID:", test_id)
print("Exact match:", image_path_lookup.get(test_id))

# Check for a matching filename if the exact lookup fails
print("Matching candidates:")
print([key for key in image_path_lookup if test_id in key][:10])

In [ ]:
target_ids = non_exact.iloc[0][["image_a", "image_b"]].tolist()

print("Target IDs:", target_ids)

display(
    embedding_df[
        embedding_df["image_id"].isin(target_ids)
    ][["image_id", "local_path"]]
)


In [ ]:
print("Available DataFrame names to check manually:")
print("Look for the DataFrame loaded from data/processed/canonical_images.parquet")

In [ ]:
import pandas as pd

manifest_path = project_root / "data/processed/canonical_images.parquet"
manifest_check = pd.read_parquet(manifest_path)

display(
    manifest_check[
        manifest_check["image_id"].isin(target_ids)
    ][["image_id", "local_path"]]
)

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
cache_dir = project_root / "data/raw/images/cache"

# Build lookup using the actual cached filenames
image_path_lookup = {
    path.stem: path
    for path in cache_dir.glob("*")
    if path.is_file()
}

def resolve_image_path(image_id):
    # Remove marketplace prefix, e.g. amazon_
    filename_stem = image_id.split("_", 1)[1]
    return image_path_lookup.get(filename_stem)

# Test the first pair
row = non_exact.iloc[0]

print("Image A:", resolve_image_path(row["image_a"]))
print("Image B:", resolve_image_path(row["image_b"]))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

top_pairs = non_exact.head(6)

for rank, (_, row) in enumerate(top_pairs.iterrows(), start=1):
    path_a = resolve_image_path(row["image_a"])
    path_b = resolve_image_path(row["image_b"])

    if path_a is None or path_b is None:
        print(f"Pair {rank}: image path not found")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(Image.open(path_a).convert("RGB"))
    axes[0].set_title(row["product_a"], fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(Image.open(path_b).convert("RGB"))
    axes[1].set_title(row["product_b"], fontsize=9)
    axes[1].axis("off")

    fig.suptitle(
        f"Pair {rank} | Cosine: {row['cosine_similarity']:.4f}"
        f" | dHash: {row['dhash_distance']}",
        fontsize=11
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

top_pairs = non_exact.head(6).copy()

evaluation_df = pd.DataFrame({
    "product_a": top_pairs["product_a"].values,
    "product_b": top_pairs["product_b"].values,
    "cosine_similarity": top_pairs["cosine_similarity"].values,
    "dhash_distance": top_pairs["dhash_distance"].values,
    "same_sha256": top_pairs["same_sha256"].values,
})

evaluation_df["manual_label"] = ""
evaluation_df["notes"] = ""

display(evaluation_df.round({"cosine_similarity": 4}))

In [ ]:
provisional_labels = [
    "uncertain",
    "different_products",
    "different_products",
    "different_products",
    "uncertain",
    "uncertain",
]

provisional_notes = [
    "Needs visual confirmation.",
    "Different tube designs and branding.",
    "Different hair products and packaging.",
    "Different printed mug designs.",
    "Needs visual confirmation.",
    "Needs visual confirmation.",
]

evaluation_df["manual_label"] = provisional_labels
evaluation_df["notes"] = provisional_notes

display(evaluation_df.round({"cosine_similarity": 4}))

In [ ]:
inspect_indices = [0, 4, 5]

for i in inspect_indices:
    row = top_pairs.iloc[i]

    path_a = resolve_image_path(row["image_a"])
    path_b = resolve_image_path(row["image_b"])

    if path_a is None or path_b is None:
        print(f"Pair {i + 1}: image path not found")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(Image.open(path_a).convert("RGB"))
    axes[0].set_title(row["product_a"], fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(Image.open(path_b).convert("RGB"))
    axes[1].set_title(row["product_b"], fontsize=9)
    axes[1].axis("off")

    fig.suptitle(
        f"Pair {i + 1} | Cosine: {row['cosine_similarity']:.4f}"
        f" | dHash: {row['dhash_distance']}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
provisional_labels = [
    "uncertain",
    "different_products",
    "different_products",
    "different_products",
    "different_products",
    "different_products",
]

provisional_notes = [
    "Similar bottle shapes; product identity unverified.",
    "Different tube designs and branding.",
    "Different hair products and packaging.",
    "Different printed mug designs.",
    "Different product packaging; high cosine but dHash distance 38.",
    "Serum and hair developer appear to be different product categories.",
]

evaluation_df["manual_label"] = provisional_labels
evaluation_df["notes"] = provisional_notes

display(evaluation_df.round({"cosine_similarity": 4}))

In [ ]:
print("Label counts:")
display(evaluation_df["manual_label"].value_counts())

print("\nSimilarity by provisional label:")
display(
    evaluation_df.groupby("manual_label").agg(
        pair_count=("manual_label", "size"),
        mean_cosine=("cosine_similarity", "mean"),
        mean_dhash=("dhash_distance", "mean"),
        min_cosine=("cosine_similarity", "min"),
        max_cosine=("cosine_similarity", "max"),
    ).round(4)
)

In [ ]:
# Create a diverse pilot sample without modifying pair_results.
# We will inspect high, middle, and lower similarity candidates.

pair_sample = pair_results[
    ~pair_results["same_sha256"]
].copy()

pair_sample = pair_sample.sort_values(
    "cosine_similarity",
    ascending=False
).reset_index(drop=True)

n = len(pair_sample)

sample_indices = sorted(set(
    [
        round(i * (n - 1) / 49)
        for i in range(50)
    ]
))

evaluation_sample = pair_sample.iloc[sample_indices].copy()
evaluation_sample["manual_label"] = ""
evaluation_sample["notes"] = ""

print("Pairs selected:", len(evaluation_sample))
display(
    evaluation_sample[
        [
            "product_a",
            "product_b",
            "cosine_similarity",
            "dhash_distance",
            "same_sha256",
            "manual_label",
            "notes",
        ]
    ].round({"cosine_similarity": 4})
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_pair_batch(evaluation_sample, start=0, batch_size=5):
    batch = evaluation_sample.iloc[start:start + batch_size]

    for idx, (_, row) in enumerate(batch.iterrows(), start=start + 1):
        path_a = resolve_image_path(row["image_a"])
        path_b = resolve_image_path(row["image_b"])

        if path_a is None or path_b is None:
            print(f"Pair {idx}: image path not found")
            continue

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

        axes[0].imshow(Image.open(path_a).convert("RGB"))
        axes[0].set_title(row["product_a"], fontsize=8)
        axes[0].axis("off")

        axes[1].imshow(Image.open(path_b).convert("RGB"))
        axes[1].set_title(row["product_b"], fontsize=8)
        axes[1].axis("off")

        fig.suptitle(
            f"Pair {idx} | Cosine: {row['cosine_similarity']:.4f}"
            f" | dHash: {row['dhash_distance']}",
            fontsize=10
        )

        plt.tight_layout()
        plt.show()

show_pair_batch(evaluation_sample, start=0, batch_size=5)

In [ ]:
show_pair_batch(evaluation_sample, start=5, batch_size=5)

In [ ]:
# Example: fill labels only after visually inspecting the pairs.
# Replace these example values with your actual observations.

evaluation_sample.loc[
    evaluation_sample.index[:5], "manual_label"
] = [
    "", "", "", "", ""
]

In [ ]:
print("Total pairs:", len(evaluation_sample))
print("\nLabel counts:")
print(evaluation_sample["manual_label"].value_counts(dropna=False))

print(
    "\nUnlabelled pairs:",
    evaluation_sample["manual_label"].isna().sum()
    + evaluation_sample["manual_label"].eq("").sum()
)


In [ ]:
import pandas as pd

label_summary = (
    evaluation_sample
    .groupby("manual_label", dropna=False)
    .agg(
        pair_count=("manual_label", "size"),
        mean_cosine=("cosine_similarity", "mean"),
        median_cosine=("cosine_similarity", "median"),
        mean_dhash=("dhash_distance", "mean"),
        median_dhash=("dhash_distance", "median"),
    )
    .round(4)
)

display(label_summary)

In [ ]:
evaluation_sample.to_csv(
    "cv_manual_evaluation_50.csv",
    index=False
)

print("Saved CV evaluation table.")

In [ ]:
show_pair_batch(evaluation_sample, start=0, batch_size=5)

In [ ]:
labels_0_4 = [
    "different_products",
    "uncertain",
    "same_product_different_view",
    "different_products",
    "different_products",
]

evaluation_sample.iloc[
    0:5,
    evaluation_sample.columns.get_loc("manual_label")
] = labels_0_4

In [ ]:
print(
    "Labelled:",
    evaluation_sample["manual_label"].isin([
        "same_image_different_encoding",
        "same_product_different_view",
        "different_products",
        "uncertain",
    ]).sum(),
    "/",
    len(evaluation_sample)
)

print(evaluation_sample["manual_label"].value_counts())

In [ ]:
show_pair_batch(evaluation_sample, start=5, batch_size=10)

In [ ]:
show_pair_batch(evaluation_sample, start=15, batch_size=5)

In [ ]:
show_pair_batch(evaluation_sample, start=20, batch_size=10)

In [ ]:
show_pair_batch(evaluation_sample, start=30, batch_size=5)

In [ ]:
valid_labels = [
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
]

print("Total pairs:", len(evaluation_sample))
print(
    "Labelled pairs:",
    evaluation_sample["manual_label"].isin(valid_labels).sum()
)
print("\nLabel distribution:")
print(evaluation_sample["manual_label"].value_counts())

In [ ]:
valid_labels = [
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
]

labelled_rows = evaluation_sample[
    evaluation_sample["manual_label"].isin(valid_labels)
]

display(
    labelled_rows[
        [
            "product_a",
            "product_b",
            "cosine_similarity",
            "dhash_distance",
            "manual_label",
            "notes",
        ]
    ]
)

In [ ]:
show_pair_batch(evaluation_sample, start=5, batch_size=5)

In [ ]:
# Replace these example blanks with your actual labels.
next_labels = [
    "", "", "", "", ""
]

evaluation_sample.iloc[
    5:10,
    evaluation_sample.columns.get_loc("manual_label")
] = next_labels

In [ ]:
show_pair_batch(evaluation_sample, start=5, batch_size=10)

In [ ]:
valid_labels = [
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
]

labelled_count = evaluation_sample["manual_label"].isin(
    valid_labels
).sum()

print(f"Labelled pairs: {labelled_count}/20")
print("\nLabel distribution:")
print(evaluation_sample["manual_label"].value_counts())

In [ ]:
show_pair_batch(evaluation_sample, start=5, batch_size=10)

In [ ]:
# Record only the pairs we have visually assessed
labels_to_save = {
    7: "different_products",  # Pair 8 (zero-based index)
    8: "different_products",  # Pair 9 (zero-based index)
}

for idx, label in labels_to_save.items():
    evaluation_sample.loc[
        evaluation_sample.index[idx], "manual_label"
    ] = label

print(
    "Labelled pairs:",
    evaluation_sample["manual_label"].isin([
        "same_image_different_encoding",
        "same_product_different_view",
        "different_products",
        "uncertain",
    ]).sum()
)

In [ ]:
show_pair_batch(evaluation_sample, start=9, batch_size=5)

In [ ]:
show_pair_batch(evaluation_sample, start=9, batch_size=5)

In [ ]:
from pathlib import Path

# Notebook is inside the notebooks/ folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

label_dir = project_root / "data" / "processed"
label_dir.mkdir(parents=True, exist_ok=True)

label_file = label_dir / "vision_manual_evaluation.csv"

# Save all 50 candidate pairs, including their current labels
evaluation_sample.to_csv(label_file, index=False)

print("Saved to:", label_file.resolve())
print("Total candidate pairs:", len(evaluation_sample))
print(
    "Labelled pairs:",
    evaluation_sample["manual_label"].isin([
        "same_image_different_encoding",
        "same_product_different_view",
        "different_products",
        "uncertain",
    ]).sum()
)

In [ ]:
def save_vision_labels():
    evaluation_sample.to_csv(label_file, index=False)

    valid_labels = {
        "same_image_different_encoding",
        "same_product_different_view",
        "different_products",
        "uncertain",
    }

    count = evaluation_sample["manual_label"].isin(valid_labels).sum()

    print(f"Saved successfully: {count}/20 target labels")
    print(f"File: {label_file.resolve()}")

In [ ]:
save_vision_labels()


In [ ]:
saved_labels = pd.read_csv(label_file)

print("Rows saved:", len(saved_labels))
print("Non-empty labels:", saved_labels["manual_label"].isin([
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
]).sum())

display(
    saved_labels.loc[
        saved_labels["manual_label"].isin([
            "same_image_different_encoding",
            "same_product_different_view",
            "different_products",
            "uncertain",
        ]),
        ["image_a", "image_b", "cosine_similarity",
         "dhash_distance", "manual_label", "notes"]
    ]
)

In [ ]:
show_pair_batch(evaluation_sample, start=0, batch_size=20)

In [ ]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Set up CSV save location
# --------------------------------------------------
project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

label_dir = project_root / "data" / "processed"
label_dir.mkdir(parents=True, exist_ok=True)

label_file = label_dir / "vision_manual_evaluation.csv"

# --------------------------------------------------
# 2. Labels assessed from your screenshots
# Pair numbers are 1-based in the screenshots.
# Dictionary keys below are 0-based DataFrame positions.
# --------------------------------------------------
labels_batch = {
    0: "uncertain",                     # Pair 1: similar bottles, identity unclear
    1: "different_products",            # Pair 2: political sign vs book
    2: "different_products",            # Pair 3: folding stool vs drawing table
    3: "different_products",            # Pair 4: stickers vs children's book
    4: "different_products",            # Pair 5: skin tape vs cushion
    5: "different_products",            # Pair 6: costume vs child's ride-on toy
    6: "different_products",            # Pair 7: dollhouse vs book cover
    7: "different_products",            # Pair 8: pink item vs watch strap
    8: "different_products",            # Pair 9: bee graphic vs fuzzy pouch
    9: "different_products",            # Pair 10: cosmetic jar vs camera lens
    10: "different_products",           # Pair 11: dental product vs book
    11: "different_products",           # Pair 12: soap gift set vs hair product
    12: "different_products",           # Pair 13: black plush toy vs book
    13: "different_products",           # Pair 14: stress toy vs resin product
    14: "different_products",           # Pair 15: cleaning accessories vs book
    15: "different_products",           # Pair 16: spray bottle vs memory card
    16: "different_products",           # Pair 17: cosmetic product vs car display
    17: "different_products",           # Pair 18: camera arm vs novelty toy
    18: "different_products",           # Pair 19: knife set vs dollhouse
    19: "different_products",           # Pair 20: dress vs book
}

VALID_LABELS = {
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
}

# --------------------------------------------------
# 3. Validate before applying labels
# --------------------------------------------------
if len(evaluation_sample) < 20:
    raise ValueError("evaluation_sample has fewer than 20 rows.")

for idx, label in labels_batch.items():
    if idx < 0 or idx >= len(evaluation_sample):
        raise IndexError(f"Invalid DataFrame position: {idx}")
    if label not in VALID_LABELS:
        raise ValueError(f"Invalid label at position {idx}: {label}")

# --------------------------------------------------
# 4. Apply labels to the first 20 sampled pairs
# --------------------------------------------------
for idx, label in labels_batch.items():
    row_index = evaluation_sample.index[idx]
    evaluation_sample.loc[row_index, "manual_label"] = label

# --------------------------------------------------
# 5. Save the complete evaluation sample
# --------------------------------------------------
evaluation_sample.to_csv(label_file, index=False)

# --------------------------------------------------
# 6. Read back and verify
# --------------------------------------------------
saved_labels = pd.read_csv(label_file)

labelled_mask = saved_labels["manual_label"].isin(VALID_LABELS)

print("Saved successfully!")
print("File:", label_file.resolve())
print("Total candidate pairs:", len(saved_labels))
print("Labelled pairs:", int(labelled_mask.sum()))
print("\nLabel distribution:")
print(saved_labels.loc[labelled_mask, "manual_label"].value_counts())

print("\nSaved labels for pairs 1–20:")
display(
    saved_labels.iloc[:20][
        [
            "image_a",
            "image_b",
            "cosine_similarity",
            "dhash_distance",
            "manual_label",
        ]
    ]
)


In [ ]:

import pandas as pd

label_file = project_root / "data" / "processed" / "vision_manual_evaluation.csv"
saved_labels = pd.read_csv(label_file)

VALID_LABELS = {
    "same_image_different_encoding",
    "same_product_different_view",
    "different_products",
    "uncertain",
}

labelled = saved_labels[
    saved_labels["manual_label"].isin(VALID_LABELS)
].copy()

print("=== MANUAL EVALUATION AUDIT ===")
print("Candidate pairs:", len(saved_labels))
print("Labelled pairs:", len(labelled))
print("Unlabelled pairs:", len(saved_labels) - len(labelled))
print("Missing image_a:", saved_labels["image_a"].isna().sum())
print("Missing image_b:", saved_labels["image_b"].isna().sum())
print("Duplicate image pairs:", saved_labels.duplicated(
    subset=["image_a", "image_b"]
).sum())

print("\nLabel distribution:")
print(labelled["manual_label"].value_counts())

print("\nSimilarity summary by manual label:")
display(
    labelled.groupby("manual_label").agg(
        pair_count=("cosine_similarity", "size"),
        mean_cosine=("cosine_similarity", "mean"),
        mean_dhash=("dhash_distance", "mean"),
    ).round(4)
)


In [ ]:

# Compare dHash and ResNet-18 on the labelled sample

import matplotlib.pyplot as plt

labelled = saved_labels[
    saved_labels["manual_label"].isin(VALID_LABELS)
].copy()

# Scatter plot: each point is one manually assessed pair
plt.figure(figsize=(9, 6))

for label, group in labelled.groupby("manual_label"):
    plt.scatter(
        group["dhash_distance"],
        group["cosine_similarity"],
        label=label,
        s=65,
        alpha=0.8
    )

plt.xlabel("dHash Hamming Distance (lower = more similar)")
plt.ylabel("ResNet-18 Cosine Similarity (higher = more similar)")
plt.title("Image Similarity Methods — Manual Evaluation Sample")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Inspect pairs with high ResNet similarity but different products
print("Different-product pairs with cosine similarity >= 0.80:")

display(
    labelled[
        (labelled["manual_label"] == "different_products") &
        (labelled["cosine_similarity"] >= 0.80)
    ][
        [
            "image_a",
            "image_b",
            "cosine_similarity",
            "dhash_distance",
            "manual_label",
        ]
    ].sort_values("cosine_similarity", ascending=False)
)


### Computer Vision Pilot — Similarity Evaluation Findings

**Objective:** Explore whether perceptual hashing (dHash) and ResNet-18 embeddings can identify potentially similar marketplace product images.

**Methodology**
- Computed 64-bit dHash distances and ResNet-18 cosine similarities.
- Manually labelled 20 candidate pairs selected across the similarity ranking.
- Saved the manual evaluation sample to `data/processed/vision_manual_evaluation.csv`.

**Observed findings**
- 19 pairs were labelled `different_products`.
- 1 pair was labelled `uncertain`.
- The different-product pairs had mean cosine similarity of 0.6110 and mean dHash distance of 28.7895.
- The uncertain pair had cosine similarity of 0.9311 and dHash distance of 4.
- No labelled different-product pair in this sample had cosine similarity ≥ 0.80.

**Interpretation and limitations**

Neither dHash distance nor ResNet-18 cosine similarity should be treated as proof of product identity. The sample is small, and the selected pairs are not a representative random sample. The results are exploratory and do not establish production precision, recall, or accuracy. The uncertain pair requires additional product-level evidence before it can be classified.

**Next:** Audit OCR coverage, packaging consistency fields, and image-download completeness.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

manifest_path = project_root / "data" / "processed" / "canonical_images.parquet"
images = pd.read_parquet(manifest_path)

print("Manifest shape:", images.shape)

# 1. OCR coverage
print("\n=== OCR COVERAGE ===")
print("Total image records:", len(images))
print("OCR text present:", images["ocr_text"].notna().sum())
print("OCR text missing:", images["ocr_text"].isna().sum())
print(
    "OCR coverage (%):",
    round(images["ocr_text"].notna().mean() * 100, 2)
)

# 2. Packaging consistency fields
score_columns = [
    "packaging_consistency_score",
    "ocr_packaging_match_score",
]

print("\n=== PACKAGING FIELD COVERAGE ===")
for col in score_columns:
    if col in images.columns:
        present = images[col].notna().sum()
        print(f"{col}: {present}/{len(images)} populated")
        if present:
            print(images[col].describe())

# 3. OCR coverage by marketplace
print("\n=== OCR BY MARKETPLACE ===")
display(
    images.groupby("marketplace").agg(
        image_records=("image_id", "size"),
        ocr_present=("ocr_text", lambda s: s.notna().sum()),
        ocr_missing=("ocr_text", lambda s: s.isna().sum()),
    )
)

# 4. Inspect actual examples, not just counts
print("\n=== OCR EXAMPLES ===")
ocr_examples = images.loc[
    images["ocr_text"].notna(),
    [
        "image_id",
        "canonical_product_id",
        "marketplace",
        "image_type",
        "ocr_text",
        "packaging_consistency_score",
        "ocr_packaging_match_score",
    ],
].head(10)

display(ocr_examples)

In [ ]:

from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Search existing source files for the relevant field names.
search_terms = [
    "packaging_consistency_score",
    "ocr_packaging_match_score",
    "packaging_consistency",
]

source_dirs = [
    project_root / "src",
    project_root / "notebooks",
]

for folder in source_dirs:
    if not folder.exists():
        continue

    for file_path in folder.rglob("*"):
        if file_path.suffix not in {".py", ".ipynb"}:
            continue

        # Avoid generated environments and cache directories.
        if any(part in {".venv", "__pycache__", ".ipynb_checkpoints"}
               for part in file_path.parts):
            continue

        try:
            content = file_path.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue

        matching_lines = [
            (line_no, line.strip())
            for line_no, line in enumerate(content.splitlines(), start=1)
            if any(term in line for term in search_terms)
        ]

        if matching_lines:
            print(f"\n--- {file_path.relative_to(project_root)} ---")
            for line_no, line in matching_lines:
                print(f"{line_no}: {line[:240]}")


In [ ]:

from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

files_to_inspect = {
    "src/vision/image_analysis.py": [(60, 165)],
    "src/risk/engine.py": [(165, 205)],
    "src/data/pipeline.py": [(195, 245)],
}

for relative_path, ranges in files_to_inspect.items():
    file_path = project_root / relative_path

    print(f"\n{'=' * 70}")
    print(relative_path)
    print("=" * 70)

    if not file_path.exists():
        print("File not found.")
        continue

    lines = file_path.read_text(encoding="utf-8").splitlines()

    for start, end in ranges:
        for line_no in range(start, min(end, len(lines)) + 1):
            print(f"{line_no:4}: {lines[line_no - 1]}")


In [ ]:

from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

source_path = project_root / "src" / "vision" / "image_analysis.py"
lines = source_path.read_text(encoding="utf-8").splitlines()

# Print the packaging consistency function and its full body
start = next(
    i for i, line in enumerate(lines)
    if line.startswith("def calculate_packaging_consistency")
)

end = start + 1

while end < len(lines):
    line = lines[end]

    # Stop at the next top-level function or class
    if line.startswith("def ") or line.startswith("class "):
        break

    end += 1

print(f"Source: {source_path}")
print(f"Lines {start + 1}–{end}")
print("\n".join(
    f"{i + 1:4}: {lines[i]}"
    for i in range(start, end)
))


In [ ]:

from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

source_path = project_root / "src" / "vision" / "image_analysis.py"
lines = source_path.read_text(encoding="utf-8").splitlines()

# Print lines 95–115 of the existing function
for i in range(94, min(115, len(lines))):
    print(f"{i + 1:4}: {lines[i]}")


In [ ]:

import pandas as pd
from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

manifest_path = project_root / "data" / "processed" / "canonical_images.parquet"
images = pd.read_parquet(manifest_path)

# Separate records with and without OCR
has_ocr = images["ocr_text"].fillna("").str.strip().ne("")
scores = images["packaging_consistency_score"]

print("=== SCORE AND OCR AUDIT ===")
print("Total records:", len(images))
print("OCR present:", int(has_ocr.sum()))
print("OCR missing:", int((~has_ocr).sum()))
print("Scores populated:", int(scores.notna().sum()))
print("Scores exactly 0.50:", int(scores.eq(0.50).sum()))

print("\n=== SCORE DISTRIBUTION BY OCR AVAILABILITY ===")
for label, mask in [
    ("OCR present", has_ocr),
    ("OCR missing", ~has_ocr),
]:
    subset = images.loc[mask, "packaging_consistency_score"]

    print(f"\n{label}")
    print("Records:", len(subset))
    print("Scores populated:", int(subset.notna().sum()))
    print("Scores equal to 0.50:", int(subset.eq(0.50).sum()))

    if subset.notna().any():
        print(subset.describe())

print("\n=== OCR-PRESENT EXAMPLES ===")
display(
    images.loc[
        has_ocr,
        [
            "image_id",
            "marketplace",
            "ocr_text",
            "packaging_consistency_score",
        ],
    ].sort_values("packaging_consistency_score").head(10)
)

print("\n=== HIGHEST-SCORE OCR EXAMPLES ===")
display(
    images.loc[
        has_ocr,
        [
            "image_id",
            "marketplace",
            "ocr_text",
            "packaging_consistency_score",
        ],
    ].sort_values("packaging_consistency_score", ascending=False).head(10)
)


In [ ]:

# Inspect the available manifest columns first
print("Manifest columns:")
print(images.columns.tolist())

# Show OCR examples with all available metadata fields
metadata_candidates = [
    "image_id",
    "canonical_product_id",
    "marketplace",
    "ocr_text",
    "packaging_consistency_score",
    "product_title",
    "title",
    "product_brand",
    "brand",
]

available_columns = [
    col for col in metadata_candidates
    if col in images.columns
]

display(
    images.loc[
        images["ocr_text"].fillna("").str.strip().ne(""),
        available_columns,
    ].head(15)
)


In [ ]:

from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Find existing structured product datasets
search_roots = [
    project_root / "data" / "processed",
    project_root / "data" / "raw",
]

candidates = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in {".parquet", ".csv", ".json"}:
            continue

        # Skip image cache and backup directories
        if any(
            part.lower() in {"cache", "images", "data_backup"}
            for part in path.parts
        ):
            continue

        candidates.append(path)

print("Potential metadata files:")
for path in candidates[:100]:
    print(path.relative_to(project_root))


In [ ]:

import pandas as pd
from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

products_path = project_root / "data" / "processed" / "canonical_products.parquet"
images_path = project_root / "data" / "processed" / "canonical_images.parquet"

products = pd.read_parquet(products_path)
images = pd.read_parquet(images_path)

print("=== CANONICAL PRODUCTS ===")
print("Shape:", products.shape)
print("Columns:")
print(products.columns.tolist())

print("\n=== PRODUCT METADATA SAMPLE ===")
display(products.head(5))

print("\n=== POSSIBLE JOIN KEYS ===")
for col in ["canonical_product_id", "product_id", "title", "product_title", "brand", "marketplace"]:
    if col in products.columns:
        print(
            f"{col}: populated={products[col].notna().sum()}, "
            f"unique={products[col].nunique(dropna=True)}"
        )


In [ ]:

# Audit the product metadata join before changing any scores.
# This cell only reads data and creates an in-memory preview.

products_path = project_root / "data" / "processed" / "canonical_products.parquet"
images_path = project_root / "data" / "processed" / "canonical_images.parquet"

products = pd.read_parquet(products_path)
images = pd.read_parquet(images_path)

product_metadata = products[
    ["canonical_product_id", "title", "brand", "marketplace"]
].copy()

# Check whether canonical product IDs have compatible types.
print("Products ID dtype:", products["canonical_product_id"].dtype)
print("Images ID dtype:", images["canonical_product_id"].dtype)

# Check image IDs against product metadata IDs.
product_ids = set(product_metadata["canonical_product_id"].dropna())
image_ids = set(images["canonical_product_id"].dropna())

print("\nProduct metadata rows:", len(product_metadata))
print("Image manifest rows:", len(images))
print("Distinct image product IDs:", len(image_ids))
print("Image product IDs found in metadata:", len(image_ids & product_ids))
print("Image product IDs missing from metadata:", len(image_ids - product_ids))

# Preview the join without modifying either source DataFrame.
image_metadata_preview = images.merge(
    product_metadata,
    on="canonical_product_id",
    how="left",
    suffixes=("_image", "_product"),
    validate="many_to_one",
    indicator=True,
)

print("\nJoin results:")
print(image_metadata_preview["_merge"].value_counts())

print("\nOCR coverage after join:")
print("OCR present:", image_metadata_preview["ocr_text"].notna().sum())
print(
    "OCR present and title available:",
    (
        image_metadata_preview["ocr_text"].notna()
        & image_metadata_preview["title"].notna()
    ).sum(),
)

display(
    image_metadata_preview.loc[
        image_metadata_preview["ocr_text"].notna(),
        [
            "canonical_product_id",
            "marketplace_image",
            "title",
            "brand",
            "ocr_text",
            "packaging_consistency_score",
        ],
    ].head(10)
)


In [ ]:

import re
import numpy as np
import pandas as pd

def audit_packaging_score(ocr_text, title, brand):
    """Reproduce the project's existing token-overlap scoring logic."""
    if not isinstance(ocr_text, str) or not ocr_text.strip():
        return 0.50

    def normalize_tokens(text):
        text = re.sub(r"[^\w\s]", " ", str(text).lower())
        return {word for word in text.split() if len(word) >= 3}

    ocr_words = normalize_tokens(ocr_text)
    reference_words = normalize_tokens(title)

    if pd.notna(brand):
        reference_words.update(normalize_tokens(brand))

    if not ocr_words or not reference_words:
        return 0.50

    overlap = ocr_words & reference_words
    return round(float(min(1.0, len(overlap) / min(
        len(ocr_words), len(reference_words)
    ))), 2)


# Use a fresh in-memory view; do not overwrite the source manifest.
ocr_audit = image_metadata_preview.loc[
    image_metadata_preview["ocr_text"].notna()
].copy()

ocr_audit["recomputed_score"] = ocr_audit.apply(
    lambda row: audit_packaging_score(
        row["ocr_text"], row["title"], row["brand"]
    ),
    axis=1,
)

ocr_audit["stored_score"] = pd.to_numeric(
    ocr_audit["packaging_consistency_score"], errors="coerce"
)

ocr_audit["score_difference"] = (
    ocr_audit["recomputed_score"] - ocr_audit["stored_score"]
).round(2)

print("OCR records audited:", len(ocr_audit))
print("Stored scores available:", ocr_audit["stored_score"].notna().sum())
print(
    "Scores matching:",
    (
        ocr_audit["recomputed_score"].eq(ocr_audit["stored_score"])
        | (
            ocr_audit["recomputed_score"].isna()
            & ocr_audit["stored_score"].isna()
        )
    ).sum(),
)
print("Scores differing:", ocr_audit["score_difference"].ne(0).sum())

display(
    ocr_audit.loc[
        ocr_audit["score_difference"].ne(0),
        [
            "canonical_product_id",
            "marketplace_image",
            "title",
            "brand",
            "ocr_text",
            "stored_score",
            "recomputed_score",
            "score_difference",
        ],
    ].head(20)
)


In [ ]:

# Inspect score discrepancies without modifying source data.

mismatches = ocr_audit.loc[
    ocr_audit["score_difference"].ne(0)
].copy()

print("Mismatched records:", len(mismatches))

print("\nDifference distribution:")
display(
    mismatches["score_difference"]
    .value_counts()
    .sort_index()
    .rename_axis("recomputed_minus_stored")
    .to_frame("count")
)

print("\nStored vs recomputed score summary:")
display(
    mismatches[
        ["stored_score", "recomputed_score", "score_difference"]
    ].describe().round(3)
)

print("\nFirst 20 mismatches:")
display(
    mismatches[
        [
            "canonical_product_id",
            "marketplace_image",
            "title",
            "brand",
            "ocr_text",
            "stored_score",
            "recomputed_score",
            "score_difference",
        ]
    ].head(20)
)


In [ ]:

# Trace the original implementation and inspect discrepancies.
# This cell reads source code and in-memory DataFrames only.

from pathlib import Path
import inspect

source_path = project_root / "src" / "vision" / "image_analysis.py"
pipeline_path = project_root / "src" / "data" / "pipeline.py"

print("=== SCORING FUNCTION IN SOURCE ===")
source_text = source_path.read_text(encoding="utf-8")
source_lines = source_text.splitlines()

for i, line in enumerate(source_lines):
    if "def calculate_packaging_consistency" in line:
        start = max(0, i - 2)
        end = min(len(source_lines), i + 35)
        print("\n".join(
            f"{j + 1}: {source_lines[j]}"
            for j in range(start, end)
        ))
        break

print("\n=== PIPELINE REFERENCES ===")
pipeline_lines = pipeline_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(pipeline_lines):
    if any(term in line.lower() for term in [
        "calculate_packaging_consistency",
        "packaging_consistency_score",
        "ocr_packaging_match_score",
    ]):
        start = max(0, i - 3)
        end = min(len(pipeline_lines), i + 5)
        print(f"\n--- Around line {i + 1} ---")
        print("\n".join(
            f"{j + 1}: {pipeline_lines[j]}"
            for j in range(start, end)
        ))

print("\n=== MISMATCH EXAMPLES: EXACT VALUES ===")
for _, row in mismatches.head(10).iterrows():
    print("\nProduct:", repr(row["canonical_product_id"]))
    print("Title:", repr(row["title"]))
    print("Brand:", repr(row["brand"]))
    print("OCR:", repr(row["ocr_text"]))
    print("Stored:", repr(row["stored_score"]))
    print("Recomputed:", repr(row["recomputed_score"]))


In [ ]:

# OCR quality and coverage audit
# Uses the existing joined image metadata.

ocr_quality = image_metadata_preview.copy()

ocr_quality["ocr_text"] = ocr_quality["ocr_text"].fillna("").astype(str)
ocr_quality["ocr_present"] = ocr_quality["ocr_text"].str.strip().ne("")
ocr_quality["ocr_char_count"] = ocr_quality["ocr_text"].str.len()

def get_ocr_token_count(text):
    return len(re.findall(r"\b\w+\b", text))

ocr_quality["ocr_token_count"] = ocr_quality["ocr_text"].apply(
    get_ocr_token_count
)

ocr_present = ocr_quality.loc[ocr_quality["ocr_present"]].copy()

print("=== OCR COVERAGE ===")
print("Total image records:", len(ocr_quality))
print("OCR-present records:", len(ocr_present))
print(
    "OCR coverage (%):",
    round(100 * len(ocr_present) / max(len(ocr_quality), 1), 2),
)

print("\n=== OCR TEXT LENGTH ===")
display(
    ocr_present[["ocr_char_count", "ocr_token_count"]]
    .describe()
    .round(2)
)

print("\n=== OCR BY MARKETPLACE ===")
display(
    ocr_quality.groupby("marketplace_image", dropna=False)
    .agg(
        total_images=("canonical_product_id", "size"),
        ocr_present=("ocr_present", "sum"),
    )
    .assign(
        ocr_coverage_pct=lambda df: (
            100 * df["ocr_present"] / df["total_images"]
        ).round(2)
    )
)

print("\n=== SHORTEST OCR TEXTS ===")
display(
    ocr_present.sort_values("ocr_char_count")[
        [
            "canonical_product_id",
            "title",
            "brand",
            "ocr_text",
            "ocr_char_count",
            "ocr_token_count",
        ]
    ].head(15)
)

print("\n=== LONGEST OCR TEXTS ===")
display(
    ocr_present.sort_values("ocr_char_count", ascending=False)[
        [
            "canonical_product_id",
            "title",
            "brand",
            "ocr_text",
            "ocr_char_count",
            "ocr_token_count",
        ]
    ].head(10)
)


In [ ]:

# Classify OCR text quality for investigation.
# This creates an in-memory audit only.

ocr_quality_audit = ocr_present.copy()

def classify_ocr_text(text):
    text = str(text).strip()

    if not text:
        return "empty"

    tokens = re.findall(r"\b\w+\b", text)
    alphabetic_tokens = [
        token for token in tokens if any(ch.isalpha() for ch in token)
    ]

    if len(text) < 3 or not alphabetic_tokens:
        return "very_short_or_nonalphabetic"

    if len(alphabetic_tokens) < 2 or len(text) < 8:
        return "limited_text_review"

    return "potentially_usable_review"

ocr_quality_audit["ocr_quality_group"] = (
    ocr_quality_audit["ocr_text"].apply(classify_ocr_text)
)

print("OCR quality group counts:")
display(
    ocr_quality_audit["ocr_quality_group"]
    .value_counts()
    .rename_axis("quality_group")
    .to_frame("records")
)

print("\nExamples from each group:")
for group, subset in ocr_quality_audit.groupby("ocr_quality_group"):
    print(f"\n--- {group} ---")
    display(
        subset[
            ["canonical_product_id", "title", "brand", "ocr_text"]
        ].head(5)
    )


In [ ]:

# OCR-to-listing agreement audit
# Read-only: creates a new in-memory diagnostic DataFrame.

def normalized_tokens(text):
    if not isinstance(text, str):
        return set()

    text = re.sub(r"[^\w\s]", " ", text.lower())
    return {token for token in text.split() if len(token) >= 3}


def brand_match_status(ocr_text, brand):
    brand_tokens = normalized_tokens(brand)
    ocr_tokens = normalized_tokens(ocr_text)

    if not brand_tokens:
        return "brand_unavailable"

    if not ocr_tokens:
        return "ocr_unusable"

    if brand_tokens.issubset(ocr_tokens):
        return "brand_tokens_match"

    if brand_tokens & ocr_tokens:
        return "partial_brand_match"

    return "no_brand_token_match"


def title_token_overlap(ocr_text, title):
    ocr_tokens = normalized_tokens(ocr_text)
    title_tokens = normalized_tokens(title)

    if not ocr_tokens or not title_tokens:
        return np.nan

    return round(
        len(ocr_tokens & title_tokens) / len(ocr_tokens), 3
    )


ocr_agreement = ocr_quality_audit.copy()

ocr_agreement["brand_match_status"] = ocr_agreement.apply(
    lambda row: brand_match_status(
        row["ocr_text"], row["brand"]
    ),
    axis=1,
)

ocr_agreement["title_token_overlap"] = ocr_agreement.apply(
    lambda row: title_token_overlap(
        row["ocr_text"], row["title"]
    ),
    axis=1,
)

print("=== BRAND AGREEMENT ===")
display(
    ocr_agreement["brand_match_status"]
    .value_counts()
    .rename_axis("brand_status")
    .to_frame("records")
)

print("\n=== TITLE TOKEN OVERLAP ===")
display(ocr_agreement["title_token_overlap"].describe().round(3))

print("\n=== EXAMPLES WITHOUT FULL BRAND TOKEN MATCH ===")
display(
    ocr_agreement.loc[
        ~ocr_agreement["brand_match_status"].eq("brand_tokens_match"),
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "brand_match_status",
            "title_token_overlap",
        ],
    ].head(20)
)


In [ ]:

# Fix the missing-column error without modifying any source data.

# Keep the original brand-mismatch records.
no_brand_match = ocr_agreement.loc[
    ocr_agreement["brand_match_status"].eq("no_brand_token_match")
].copy()

# Add score columns from the earlier OCR audit using the product ID.
score_columns = ocr_audit[
    ["canonical_product_id", "stored_score", "recomputed_score"]
].drop_duplicates(subset=["canonical_product_id"])

no_brand_match = no_brand_match.merge(
    score_columns,
    on="canonical_product_id",
    how="left",
    validate="many_to_one",
)

no_brand_match["ocr_token_count"] = no_brand_match["ocr_text"].apply(
    lambda text: len(normalized_tokens(text))
)

print("No-brand-match records:", len(no_brand_match))

display(
    no_brand_match[
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "ocr_token_count",
            "title_token_overlap",
            "stored_score",
            "recomputed_score",
        ]
    ].head(25)
)


In [ ]:

# Summarize OCR agreement patterns without changing source data.

ocr_agreement["title_overlap_group"] = pd.cut(
    ocr_agreement["title_token_overlap"],
    bins=[-0.001, 0, 0.25, 0.50, 0.75, 1.001],
    labels=[
        "zero_overlap",
        "low_overlap",
        "moderate_overlap",
        "high_overlap",
        "complete_token_overlap",
    ],
    include_lowest=True,
)

print("=== BRAND STATUS x TITLE OVERLAP ===")
display(
    pd.crosstab(
        ocr_agreement["brand_match_status"],
        ocr_agreement["title_overlap_group"],
        dropna=False,
    )
)

print("\n=== HIGH TITLE OVERLAP, NO BRAND MATCH ===")
display(
    ocr_agreement.loc[
        ocr_agreement["brand_match_status"].eq("no_brand_token_match")
        & ocr_agreement["title_token_overlap"].ge(0.75),
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "title_token_overlap",
        ],
    ].head(15)
)

print("\n=== ZERO TITLE OVERLAP, NO BRAND MATCH ===")
display(
    ocr_agreement.loc[
        ocr_agreement["brand_match_status"].eq("no_brand_token_match")
        & ocr_agreement["title_token_overlap"].eq(0),
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "title_token_overlap",
        ],
    ].head(15)
)


In [ ]:

# Create preliminary OCR review categories.
# Diagnostic only; does not modify source data or assign fraud labels.

def assign_ocr_review_category(row):
    brand_status = row["brand_match_status"]
    overlap = row["title_token_overlap"]
    quality = row["ocr_quality_group"]

    if quality == "very_short_or_nonalphabetic":
        return "insufficient_text"

    if pd.isna(overlap):
        return "insufficient_comparison"

    if brand_status == "brand_tokens_match" and overlap >= 0.25:
        return "brand_and_title_support"

    if brand_status == "partial_brand_match" and overlap >= 0.25:
        return "partial_brand_and_title_support"

    if overlap >= 0.75:
        return "title_support_without_full_brand_match"

    if overlap == 0:
        return "no_title_overlap_review"

    return "mixed_or_weak_text_evidence"


ocr_review = ocr_agreement.copy()

ocr_review["review_category"] = ocr_review.apply(
    assign_ocr_review_category, axis=1
)

print("=== PRELIMINARY REVIEW CATEGORIES ===")
display(
    ocr_review["review_category"]
    .value_counts()
    .rename_axis("review_category")
    .to_frame("records")
)

print("\n=== CATEGORY BY BRAND STATUS ===")
display(
    pd.crosstab(
        ocr_review["review_category"],
        ocr_review["brand_match_status"],
    )
)

print("\n=== EXAMPLES TO REVIEW ===")
display(
    ocr_review[
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "brand_match_status",
            "title_token_overlap",
            "review_category",
        ]
    ].head(20)
)


In [ ]:

# Select up to 3 records per review category.
# Read-only: no source files or datasets are modified.

validation_sample = (
    ocr_review
    .groupby("review_category", group_keys=False)
    .sample(frac=1, random_state=42)
    .groupby("review_category", group_keys=False)
    .head(3)
    .drop_duplicates(subset=["canonical_product_id"])
    .head(20)
    .copy()
)

print("Validation sample size:", len(validation_sample))
print("\nRecords per category:")
display(validation_sample["review_category"].value_counts())

display(
    validation_sample[
        [
            "canonical_product_id",
            "brand",
            "title",
            "ocr_text",
            "brand_match_status",
            "title_token_overlap",
            "review_category",
        ]
    ].reset_index(drop=True)
)


In [ ]:
from pathlib import Path
import pandas as pd

# Match sampled product IDs to the existing image manifest.
sample_ids = validation_sample["canonical_product_id"].unique()

sample_images = images.loc[
    images["canonical_product_id"].isin(sample_ids)
].copy()

def check_local_image(path):
    if not isinstance(path, str) or not path.strip():
        return False
    return Path(path).is_file()

sample_images["local_file_exists"] = (
    sample_images["local_path"].apply(check_local_image)
)

print("Unique sampled products:", len(sample_ids))
print("Image records for sampled products:", len(sample_images))
print(
    "Image records with local files:",
    int(sample_images["local_file_exists"].sum()),
)
print(
    "Sampled products with at least one local image:",
    sample_images.loc[
        sample_images["local_file_exists"], "canonical_product_id"
    ].nunique(),
)

display(
    sample_images[
        [
            "canonical_product_id",
            "marketplace",
            "image_type",
            "local_path",
            "local_file_exists",
        ]
    ].head(25)
)


In [ ]:

from pathlib import Path
import pandas as pd

def resolve_image_path(path):
    if not isinstance(path, str) or not path.strip():
        return None

    # Normalize Windows separators so relative paths resolve consistently.
    normalized = path.replace("\\", "/")
    candidate = Path(normalized)

    if not candidate.is_absolute():
        candidate = project_root / candidate

    if candidate.is_file():
        return candidate.resolve()

    return None


sample_images["resolved_path"] = sample_images["local_path"].apply(
    resolve_image_path
)
sample_images["local_file_exists"] = sample_images["resolved_path"].notna()

print("Image records for sampled products:", len(sample_images))
print("Image records with resolved local files:",
      int(sample_images["local_file_exists"].sum()))
print("Sampled products with at least one local image:",
      sample_images.loc[
          sample_images["local_file_exists"], "canonical_product_id"
      ].nunique())

display(
    sample_images[
        [
            "canonical_product_id",
            "image_type",
            "local_path",
            "resolved_path",
            "local_file_exists",
        ]
    ].head(25)
)


In [ ]:

from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import math

# One existing local primary image per sampled product.
primary_images = (
    sample_images.loc[
        sample_images["image_type"].eq("primary")
        & sample_images["local_file_exists"]
    ]
    .drop_duplicates(subset=["canonical_product_id"])
    .copy()
)

thumb_w, thumb_h = 180, 180
label_h = 55
columns = 4
rows = math.ceil(len(primary_images) / columns)

sheet = Image.new(
    "RGB",
    (columns * thumb_w, rows * (thumb_h + label_h)),
    "white",
)
draw = ImageDraw.Draw(sheet)

for i, (_, row) in enumerate(primary_images.iterrows()):
    try:
        with Image.open(row["resolved_path"]) as img:
            img = img.convert("RGB")
            img.thumbnail((thumb_w - 12, thumb_h - 12))

        x = (i % columns) * thumb_w
        y = (i // columns) * (thumb_h + label_h)

        sheet.paste(
            img,
            (
                x + (thumb_w - img.width) // 2,
                y + (thumb_h - img.height) // 2,
            ),
        )

        label = str(row["canonical_product_id"])
        draw.text((x + 4, y + thumb_h + 3), label[:27], fill="black")

    except Exception as exc:
        print("Could not open image:", row["canonical_product_id"], exc)

print("Primary images in contact sheet:", len(primary_images))
display(sheet)


In [ ]:

# Prepare a manual visual-validation sheet.
# This creates an in-memory table only.

validation_sheet = validation_sample.merge(
    primary_images[
        ["canonical_product_id", "resolved_path"]
    ].drop_duplicates("canonical_product_id"),
    on="canonical_product_id",
    how="left",
    validate="one_to_one",
)

validation_sheet = validation_sheet[
    [
        "canonical_product_id",
        "brand",
        "title",
        "ocr_text",
        "brand_match_status",
        "title_token_overlap",
        "review_category",
        "resolved_path",
    ]
].head(10).copy()

# Blank fields for later manual annotation.
validation_sheet["visual_text_readable"] = ""
validation_sheet["visible_brand_agrees"] = ""
validation_sheet["listing_image_relevant"] = ""
validation_sheet["review_notes"] = ""

display(validation_sheet)


In [ ]:

# Check whether the visual-validation sample has been annotated.
# This does not modify or save any files.

annotation_columns = [
    "visual_text_readable",
    "visible_brand_agrees",
    "listing_image_relevant",
]

for col in annotation_columns:
    if col not in validation_sheet.columns:
        validation_sheet[col] = ""

for col in annotation_columns:
    validation_sheet[col] = (
        validation_sheet[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

print("Validation rows:", len(validation_sheet))

for col in annotation_columns:
    print(f"\n{col}:")
    print(
        validation_sheet[col]
        .replace("", "not_annotated")
        .value_counts()
        .to_string()
    )

print("\nUnannotated rows:")
display(
    validation_sheet.loc[
        validation_sheet[annotation_columns].eq("").any(axis=1),
        [
            "canonical_product_id",
            "brand",
            "ocr_text",
            *annotation_columns,
        ],
    ]
)


In [ ]:
# ============================================================
# CV AUDIT: OCR PACKAGING CONSISTENCY SCORE VALIDATION
# Self-contained | Read-only | No source data overwritten
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# 1. Locate project and load canonical datasets
# ------------------------------------------------------------

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

products_path = (
    project_root / "data" / "processed" / "canonical_products.parquet"
)

images_path = (
    project_root / "data" / "processed" / "canonical_images.parquet"
)

products = pd.read_parquet(products_path)
images = pd.read_parquet(images_path)

print("=" * 60)
print("DATA LOADING")
print("=" * 60)

print("Products:", products.shape)
print("Images:", images.shape)

# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_product_columns = [
    "canonical_product_id",
    "title",
    "brand",
    "marketplace"
]

required_image_columns = [
    "canonical_product_id",
    "ocr_text",
    "packaging_consistency_score"
]

for col in required_product_columns:
    if col not in products.columns:
        raise ValueError(f"Missing product column: {col}")

for col in required_image_columns:
    if col not in images.columns:
        raise ValueError(f"Missing image column: {col}")

if products["canonical_product_id"].duplicated().any():
    raise ValueError("Product IDs are not unique. Stop and investigate.")

# ------------------------------------------------------------
# 3. Join image records with product metadata
# ------------------------------------------------------------

product_metadata = products[
    required_product_columns
].copy()

image_metadata_preview = images.merge(
    product_metadata,
    on="canonical_product_id",
    how="left",
    validate="many_to_one",
    suffixes=("_image", "_product"),
    indicator=True
)

print("\n" + "=" * 60)
print("METADATA JOIN")
print("=" * 60)

print("Image records:", len(images))
print("Matched:", (image_metadata_preview["_merge"] == "both").sum())
print("Unmatched:", (image_metadata_preview["_merge"] == "left_only").sum())

# ------------------------------------------------------------
# 4. Reproduce the existing packaging score calculation
# ------------------------------------------------------------

def normalize_tokens(text):
    if not isinstance(text, str):
        return set()

    text = re.sub(r"[^\w\s]", " ", text.lower())

    return {
        word for word in text.split()
        if len(word) >= 3
    }


def audit_packaging_score(ocr_text, title, brand):

    if not isinstance(ocr_text, str) or not ocr_text.strip():
        return 0.50

    ocr_words = normalize_tokens(ocr_text)

    reference_words = normalize_tokens(title)

    if pd.notna(brand):
        reference_words.update(normalize_tokens(brand))

    if not ocr_words or not reference_words:
        return 0.50

    overlap = ocr_words.intersection(reference_words)

    score = len(overlap) / min(
        len(ocr_words),
        len(reference_words)
    )

    return round(float(min(1.0, score)), 2)


# ------------------------------------------------------------
# 5. Prepare OCR audit dataset
# ------------------------------------------------------------

ocr_audit = image_metadata_preview.loc[
    image_metadata_preview["ocr_text"].notna()
    & image_metadata_preview["ocr_text"].astype(str).str.strip().ne("")
].copy()

ocr_audit["recomputed_score"] = ocr_audit.apply(
    lambda row: audit_packaging_score(
        row["ocr_text"],
        row["title"],
        row["brand"]
    ),
    axis=1
)

ocr_audit["stored_score"] = pd.to_numeric(
    ocr_audit["packaging_consistency_score"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. Compare stored and recomputed scores
# ------------------------------------------------------------

ocr_audit["score_difference"] = (
    ocr_audit["recomputed_score"]
    - ocr_audit["stored_score"]
).round(2)

valid_comparisons = ocr_audit.dropna(
    subset=["stored_score", "recomputed_score"]
).copy()

matching = np.isclose(
    valid_comparisons["stored_score"],
    valid_comparisons["recomputed_score"],
    atol=0.005
)

mismatches = valid_comparisons.loc[~matching].copy()

# ------------------------------------------------------------
# 7. Final audit report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("OCR PACKAGING SCORE AUDIT RESULTS")
print("=" * 60)

print("OCR records audited:", len(ocr_audit))

print(
    "Stored scores available:",
    ocr_audit["stored_score"].notna().sum()
)

print("Valid score comparisons:", len(valid_comparisons))

print("Matching scores:", int(matching.sum()))

print("Different scores:", len(mismatches))

print(
    "Missing stored scores:",
    ocr_audit["stored_score"].isna().sum()
)

if len(valid_comparisons) > 0:

    print(
        "\nScore agreement:",
        round(100 * matching.mean(), 2),
        "%"
    )

    print("\nScore comparison statistics:")

    display(
        valid_comparisons[
            [
                "stored_score",
                "recomputed_score",
                "score_difference"
            ]
        ].describe().round(3)
    )

    print("\nMismatch examples:")

    display(
        mismatches[
            [
                "canonical_product_id",
                "marketplace_image",
                "title",
                "brand",
                "ocr_text",
                "stored_score",
                "recomputed_score",
                "score_difference"
            ]
        ].head(20)
    )

else:
    print("No valid score comparisons available.")

print("\nAUDIT COMPLETED.")

In [ ]:
# ============================================================
# EXACT SOURCE FUNCTION VS STORED OCR SCORES
# READ-ONLY DIAGNOSTIC
# ============================================================

from pathlib import Path
import sys
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Automatically locate project root
# ------------------------------------------------------------

current_path = Path.cwd()

project_root = None

for candidate in [current_path, *current_path.parents]:
    if (candidate / "src" / "vision" / "image_analysis.py").is_file():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate src/vision/image_analysis.py. "
        "Check that the notebook is inside your project."
    )

# Add project root to Python import path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

# ------------------------------------------------------------
# 2. Import actual production scoring function
# ------------------------------------------------------------

from src.vision.image_analysis import calculate_packaging_consistency

print("Original scoring function imported successfully.")

# ------------------------------------------------------------
# 3. Load canonical datasets
# ------------------------------------------------------------

products_path = project_root / "data" / "processed" / "canonical_products.parquet"
images_path = project_root / "data" / "processed" / "canonical_images.parquet"

products = pd.read_parquet(products_path)
images = pd.read_parquet(images_path)

# ------------------------------------------------------------
# 4. Validate and join metadata
# ------------------------------------------------------------

required_product_columns = [
    "canonical_product_id",
    "title",
    "brand"
]

required_image_columns = [
    "canonical_product_id",
    "ocr_text",
    "packaging_consistency_score"
]

assert all(c in products.columns for c in required_product_columns)
assert all(c in images.columns for c in required_image_columns)

score_audit = images.merge(
    products[required_product_columns],
    on="canonical_product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nMatched records:", (score_audit["_merge"] == "both").sum())
print("Unmatched records:", (score_audit["_merge"] == "left_only").sum())

# ------------------------------------------------------------
# 5. Keep records with actual OCR text
# ------------------------------------------------------------

score_audit = score_audit.loc[
    score_audit["ocr_text"].notna()
    & score_audit["ocr_text"].astype(str).str.strip().ne("")
].copy()

# ------------------------------------------------------------
# 6. Recalculate using actual production function
# ------------------------------------------------------------

score_audit["source_recomputed_score"] = score_audit.apply(
    lambda row: calculate_packaging_consistency(
        row["ocr_text"],
        row["title"],
        row["brand"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 7. Compare both stored score columns
# ------------------------------------------------------------

for col in [
    "packaging_consistency_score",
    "ocr_packaging_match_score"
]:

    print("\n" + "=" * 60)
    print("STORED COLUMN:", col)
    print("=" * 60)

    if col not in score_audit.columns:
        print("Column not available.")
        continue

    stored = pd.to_numeric(score_audit[col], errors="coerce")
    valid = stored.notna()

    matching = np.isclose(
        stored[valid],
        score_audit.loc[valid, "source_recomputed_score"],
        atol=0.005
    )

    print("Available scores:", int(valid.sum()))
    print("Missing scores:", int((~valid).sum()))
    print("Matching:", int(matching.sum()))
    print("Different:", int((~matching).sum()))

    if valid.sum():
        print(
            "Agreement:",
            round(100 * matching.mean(), 2),
            "%"
        )

# ------------------------------------------------------------
# 8. Inspect mismatches
# ------------------------------------------------------------

score_audit["stored_score"] = pd.to_numeric(
    score_audit["packaging_consistency_score"],
    errors="coerce"
)

score_audit["score_difference"] = (
    score_audit["source_recomputed_score"]
    - score_audit["stored_score"]
).round(2)

mismatches = score_audit.loc[
    score_audit["score_difference"].abs() > 0.005
].copy()

print("\nTotal mismatches:", len(mismatches))

display(
    mismatches[
        [
            "canonical_product_id",
            "title",
            "brand",
            "ocr_text",
            "packaging_consistency_score",
            "ocr_packaging_match_score",
            "source_recomputed_score",
            "score_difference"
        ]
    ].head(20)
)

print("\nAUDIT COMPLETED — NO DATASETS MODIFIED.")

In [ ]:
# ============================================================
# ORIGINAL IMAGE MANIFEST SCORE PROVENANCE AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from src.vision.image_analysis import calculate_packaging_consistency

# ------------------------------------------------------------
# 1. Load original manifest
# ------------------------------------------------------------

manifest_path = project_root / "data" / "interim" / "image_manifest.parquet"

if not manifest_path.exists():
    raise FileNotFoundError("Original image manifest not found.")

manifest = pd.read_parquet(manifest_path)

print("Manifest shape:", manifest.shape)
print("\nManifest columns:")
print(manifest.columns.tolist())

# ------------------------------------------------------------
# 2. Validate required manifest columns
# ------------------------------------------------------------

required_manifest = [
    "image_id",
    "canonical_product_id",
    "title",
    "brand",
    "ocr_text",
    "packaging_consistency_score"
]

missing = [
    col for col in required_manifest
    if col not in manifest.columns
]

if missing:
    raise ValueError(f"Missing manifest columns: {missing}")

if manifest["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs found in manifest.")

# ------------------------------------------------------------
# 3. Prepare original manifest records
# ------------------------------------------------------------

original = manifest[required_manifest].copy()

original = original.rename(columns={
    "canonical_product_id": "manifest_product_id",
    "title": "original_title",
    "brand": "original_brand",
    "ocr_text": "original_ocr_text",
    "packaging_consistency_score": "original_stored_score"
})

# ------------------------------------------------------------
# 4. Attach canonical image records
# ------------------------------------------------------------

traced = images.merge(
    original,
    on="image_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_canonical", "_manifest")
)

print("\nManifest-image matches:", len(traced))
print("Canonical image records:", len(images))

# ------------------------------------------------------------
# 5. Attach current canonical product metadata
# ------------------------------------------------------------

product_metadata = products[
    ["canonical_product_id", "title", "brand"]
].copy()

traced = traced.merge(
    product_metadata,
    on="canonical_product_id",
    how="left",
    validate="many_to_one"
)

traced = traced.rename(columns={
    "title": "canonical_title",
    "brand": "canonical_brand"
})

# ------------------------------------------------------------
# 6. Filter records with original OCR text
# ------------------------------------------------------------

traced = traced.loc[
    traced["original_ocr_text"].notna()
    & traced["original_ocr_text"].astype(str).str.strip().ne("")
].copy()

# ------------------------------------------------------------
# 7. Recalculate using original metadata
# ------------------------------------------------------------

traced["manifest_recomputed_score"] = traced.apply(
    lambda row: calculate_packaging_consistency(
        row["original_ocr_text"],
        row["original_title"],
        row["original_brand"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 8. Recalculate using canonical metadata
# ------------------------------------------------------------

traced["canonical_recomputed_score"] = traced.apply(
    lambda row: calculate_packaging_consistency(
        row["ocr_text"],
        row["canonical_title"],
        row["canonical_brand"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 9. Compare original stored scores
# ------------------------------------------------------------

traced["original_stored_score"] = pd.to_numeric(
    traced["original_stored_score"],
    errors="coerce"
)

valid = traced["original_stored_score"].notna()

original_match = np.isclose(
    traced.loc[valid, "original_stored_score"],
    traced.loc[valid, "manifest_recomputed_score"],
    atol=0.005
)

print("\n" + "=" * 60)
print("ORIGINAL SCORE VALIDATION")
print("=" * 60)

print("OCR records:", len(traced))
print("Stored scores available:", int(valid.sum()))
print("Matches:", int(original_match.sum()))
print("Mismatches:", int((~original_match).sum()))

if valid.sum():
    print(
        "Agreement:",
        round(original_match.mean() * 100, 2),
        "%"
    )

# ------------------------------------------------------------
# 10. Compare original and canonical metadata
# ------------------------------------------------------------

traced["title_changed"] = (
    traced["original_title"].fillna("").astype(str)
    != traced["canonical_title"].fillna("").astype(str)
)

traced["brand_changed"] = (
    traced["original_brand"].fillna("").astype(str)
    != traced["canonical_brand"].fillna("").astype(str)
)

print("\n" + "=" * 60)
print("METADATA DIFFERENCES")
print("=" * 60)

print("Different titles:", int(traced["title_changed"].sum()))
print("Different brands:", int(traced["brand_changed"].sum()))

# ------------------------------------------------------------
# 11. Show detailed comparison
# ------------------------------------------------------------

display(
    traced[
        [
            "image_id",
            "canonical_product_id",
            "original_title",
            "canonical_title",
            "original_brand",
            "canonical_brand",
            "original_stored_score",
            "manifest_recomputed_score",
            "canonical_recomputed_score",
            "title_changed",
            "brand_changed"
        ]
    ].head(25)
)

print("\nPROVENANCE AUDIT COMPLETED.")

In [ ]:
# ============================================================
# DEEP OCR SCORE DISCREPANCY DIAGNOSTIC
# READ-ONLY
# ============================================================

import pandas as pd
import numpy as np
import inspect

# 1. Inspect the currently imported scoring function
print("=" * 65)
print("CURRENT PRODUCTION SCORING FUNCTION")
print("=" * 65)

print(inspect.getsource(calculate_packaging_consistency))

# 2. Compare original and canonical OCR text
traced["ocr_text_changed"] = (
    traced["original_ocr_text"].fillna("").astype(str)
    != traced["ocr_text"].fillna("").astype(str)
)

# 3. Compare original and canonical metadata
traced["title_changed"] = (
    traced["original_title"].fillna("").astype(str)
    != traced["canonical_title"].fillna("").astype(str)
)

traced["brand_changed"] = (
    traced["original_brand"].fillna("").astype(str)
    != traced["canonical_brand"].fillna("").astype(str)
)

# 4. Calculate original score differences
traced["original_score_difference"] = (
    traced["manifest_recomputed_score"]
    - traced["original_stored_score"]
).round(4)

traced["original_score_mismatch"] = ~np.isclose(
    traced["manifest_recomputed_score"],
    traced["original_stored_score"],
    atol=0.005
)

# 5. Summarize discrepancies
mismatch_rows = traced.loc[
    traced["original_score_mismatch"]
].copy()

print("\n" + "=" * 65)
print("DISCREPANCY SUMMARY")
print("=" * 65)

print("Total OCR records:", len(traced))
print("Score mismatches:", len(mismatch_rows))

print("\nAcross all OCR records:")
print("Changed titles:", int(traced["title_changed"].sum()))
print("Changed brands:", int(traced["brand_changed"].sum()))
print("Changed OCR text:", int(traced["ocr_text_changed"].sum()))

print("\nAmong the 70 score mismatches:")
print(
    "Changed titles:",
    int(mismatch_rows["title_changed"].sum())
)
print(
    "Changed brands:",
    int(mismatch_rows["brand_changed"].sum())
)
print(
    "Changed OCR text:",
    int(mismatch_rows["ocr_text_changed"].sum())
)

# 6. Check whether original and canonical score values differ
print("\nStored score vs canonical score:")

display(
    traced[
        [
            "original_stored_score",
            "packaging_consistency_score",
            "manifest_recomputed_score",
            "canonical_recomputed_score"
        ]
    ].describe().round(3)
)

# 7. Inspect mismatches in detail
print("\n" + "=" * 65)
print("FIRST 20 SCORE MISMATCHES")
print("=" * 65)

display(
    mismatch_rows[
        [
            "image_id",
            "canonical_product_id",
            "original_title",
            "canonical_title",
            "original_brand",
            "canonical_brand",
            "original_ocr_text",
            "ocr_text",
            "original_stored_score",
            "manifest_recomputed_score",
            "canonical_recomputed_score",
            "original_score_difference",
            "title_changed",
            "brand_changed",
            "ocr_text_changed"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# FINAL OCR DISCREPANCY BREAKDOWN
# ============================================================

import numpy as np

traced["score_mismatch"] = ~np.isclose(
    traced["original_stored_score"],
    traced["manifest_recomputed_score"],
    atol=0.005
)

print("TOTAL RECORDS:", len(traced))
print("TOTAL MISMATCHES:", traced["score_mismatch"].sum())

print("\nMETADATA CHANGES — ALL RECORDS")
print("Title changed:", traced["title_changed"].sum())
print("Brand changed:", traced["brand_changed"].sum())
print("OCR changed:", traced["ocr_text_changed"].sum())

m = traced[traced["score_mismatch"]].copy()

print("\nMETADATA CHANGES — MISMATCH RECORDS")
print("Title changed:", m["title_changed"].sum())
print("Brand changed:", m["brand_changed"].sum())
print("OCR changed:", m["ocr_text_changed"].sum())

print("\nMISMATCH EXAMPLES")

display(
    m[[
        "canonical_product_id",
        "original_title",
        "original_brand",
        "original_ocr_text",
        "original_stored_score",
        "manifest_recomputed_score",
        "original_score_difference",
        "title_changed",
        "brand_changed"
    ]].head(20).to_string(index=False)
)

In [ ]:
from pathlib import Path
import os

print("CURRENT WORKING DIRECTORY:")
print(Path.cwd())

print("\nFILES AND FOLDERS HERE:")
for item in Path.cwd().iterdir():
    print(item.name, "/" if item.is_dir() else "")

print("\nCHECKING PROJECT DATA FOLDERS:")

for folder in [
    Path("data"),
    Path("../data"),
    Path("../../data"),
    Path("notebooks/data")
]:
    print(f"{folder}:", "EXISTS" if folder.exists() else "NOT FOUND")

In [ ]:
from pathlib import Path

# Move one level up: notebooks -> project root
PROJECT_ROOT = Path.cwd().parent

manifest_path = PROJECT_ROOT / "data" / "interim" / "image_manifest.parquet"
products_path = PROJECT_ROOT / "data" / "processed" / "canonical_products.parquet"

print("PROJECT ROOT:", PROJECT_ROOT)

print("\nMANIFEST:")
print(manifest_path)
print("EXISTS:", manifest_path.exists())

print("\nCANONICAL PRODUCTS:")
print(products_path)
print("EXISTS:", products_path.exists())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent

manifest_path = PROJECT_ROOT / "data/interim/image_manifest.parquet"
products_path = PROJECT_ROOT / "data/processed/canonical_products.parquet"

# Load existing files
manifest = pd.read_parquet(manifest_path)
canonical_products = pd.read_parquet(products_path)

print("Manifest shape:", manifest.shape)
print("Canonical products shape:", canonical_products.shape)

# Keep only records with actual OCR text
ocr_records = manifest[
    manifest["ocr_text"].notna()
    & manifest["ocr_text"].astype(str).str.strip().ne("")
].copy()

# Select product metadata
product_metadata = canonical_products[
    ["canonical_product_id", "title", "brand"]
].copy()

# Confirm one canonical metadata row per product
assert not product_metadata["canonical_product_id"].duplicated().any()

# Join original manifest records with current product metadata
traced = ocr_records.merge(
    product_metadata,
    on="canonical_product_id",
    how="left",
    suffixes=("_original", "_current"),
    validate="many_to_one",
    indicator=True
)

# Prepare audit columns
traced = traced.rename(columns={
    "title_original": "original_title",
    "brand_original": "original_brand",
    "title_current": "current_title",
    "brand_current": "current_brand",
    "ocr_text": "original_ocr_text",
    "packaging_consistency_score": "original_stored_score"
})

traced["title_changed"] = (
    traced["original_title"].fillna("").astype(str)
    != traced["current_title"].fillna("").astype(str)
)

traced["brand_changed"] = (
    traced["original_brand"].fillna("").astype(str)
    != traced["current_brand"].fillna("").astype(str)
)

# Audit summary
print("\n--- OCR AUDIT RECONSTRUCTION ---")
print("Total OCR records:", len(traced))
print("Matched products:", (traced["_merge"] == "both").sum())
print("Unmatched products:", (traced["_merge"] != "both").sum())
print("Title changes:", traced["title_changed"].sum())
print("Brand changes:", traced["brand_changed"].sum())

display(traced[[
    "canonical_product_id",
    "original_title",
    "current_title",
    "original_brand",
    "current_brand",
    "original_ocr_text",
    "original_stored_score"
]].head())

In [ ]:
# ============================================================
# HISTORICAL OCR SCORE REPRODUCTION
# Original implementation from Git commit 361f382
# ============================================================

import numpy as np

def historical_packaging_score(ocr_text, title, brand):

    if not ocr_text or not ocr_text.strip():
        return 0.50

    ocr_tokens = set(ocr_text.lower().split())
    ref_tokens = set(str(title).lower().split())

    if brand:
        ref_tokens.update(str(brand).lower().split())

    ocr_words = {w for w in ocr_tokens if len(w) >= 3}
    ref_words = {w for w in ref_tokens if len(w) >= 3}

    if not ocr_words or not ref_words:
        return 0.50

    overlap = ocr_words.intersection(ref_words)

    score = len(overlap) / min(len(ocr_words), len(ref_words))

    return round(float(min(1.0, score)), 2)


# Recalculate using ORIGINAL manifest inputs
traced["historical_recomputed_score"] = traced.apply(
    lambda r: historical_packaging_score(
        r["original_ocr_text"],
        r["original_title"],
        r["original_brand"]
    ),
    axis=1
)

# Compare with original stored scores
traced["historical_match"] = np.isclose(
    traced["original_stored_score"],
    traced["historical_recomputed_score"],
    atol=0.005
)

# Summary
total = len(traced)
matches = int(traced["historical_match"].sum())
mismatches = total - matches

print("=" * 55)
print("HISTORICAL OCR SCORE VALIDATION")
print("=" * 55)

print("Total records:", total)
print("Historical matches:", matches)
print("Historical mismatches:", mismatches)
print("Historical agreement:", round(matches / total * 100, 2), "%")

# Show remaining mismatches
print("\nREMAINING MISMATCH EXAMPLES:")

display(
    traced.loc[
        ~traced["historical_match"],
        [
            "canonical_product_id",
            "original_title",
            "original_brand",
            "original_ocr_text",
            "original_stored_score",
            "historical_recomputed_score"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# INVESTIGATION OF 11 HISTORICAL SCORE MISMATCHES
# ============================================================

mismatches = traced.loc[
    ~traced["historical_match"]
].copy()

print("TOTAL REMAINING MISMATCHES:", len(mismatches))

# Show complete original inputs and scores
columns = [
    "canonical_product_id",
    "original_title",
    "original_brand",
    "original_ocr_text",
    "original_stored_score",
    "historical_recomputed_score",
    "current_title",
    "current_brand"
]

display(mismatches[columns].to_string(index=False))

In [ ]:
# ============================================================
# TOKEN-LEVEL SCORING BREAKDOWN
# ============================================================

def inspect_historical_tokens(row):

    ocr_tokens = set(str(row["original_ocr_text"]).lower().split())

    ref_tokens = set(str(row["original_title"]).lower().split())

    if pd.notna(row["original_brand"]):
        ref_tokens.update(str(row["original_brand"]).lower().split())

    ocr_words = {w for w in ocr_tokens if len(w) >= 3}
    ref_words = {w for w in ref_tokens if len(w) >= 3}

    overlap = ocr_words.intersection(ref_words)

    denominator = min(len(ocr_words), len(ref_words))

    raw_score = (
        len(overlap) / denominator
        if denominator else None
    )

    return pd.Series({
        "ocr_token_count": len(ocr_words),
        "reference_token_count": len(ref_words),
        "overlap_count": len(overlap),
        "overlap_tokens": sorted(overlap),
        "raw_score": raw_score,
        "rounded_score": round(raw_score, 2) if raw_score is not None else 0.50
    })


token_audit = mismatches.apply(inspect_historical_tokens, axis=1)

mismatch_analysis = pd.concat(
    [
        mismatches[[
            "canonical_product_id",
            "original_stored_score",
            "historical_recomputed_score"
        ]],
        token_audit
    ],
    axis=1
)

display(mismatch_analysis)

In [ ]:
# ============================================================
# CV IMAGE INTEGRITY AUDIT — DATASET LEVEL
# ============================================================

canonical_images_path = (
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)

assert canonical_images_path.exists(), (
    f"Missing canonical image dataset: {canonical_images_path}"
)

canonical_images = pd.read_parquet(canonical_images_path)

print("=" * 60)
print("CANONICAL IMAGE INTEGRITY AUDIT")
print("=" * 60)

print("\nDATASET SHAPE:", canonical_images.shape)
print("UNIQUE IMAGE IDs:", canonical_images["image_id"].nunique())
print(
    "UNIQUE PRODUCT IDs:",
    canonical_images["canonical_product_id"].nunique()
)

print(
    "DUPLICATE IMAGE IDs:",
    canonical_images["image_id"].duplicated().sum()
)

# ------------------------------------------------
# 1. MISSING VALUES
# ------------------------------------------------

print("\n--- MISSING VALUES ---")

missing_report = (
    canonical_images.isna()
    .sum()
    .to_frame("missing_count")
)

missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(canonical_images) * 100
).round(2)

display(missing_report.sort_values(
    "missing_percentage",
    ascending=False
))

# ------------------------------------------------
# 2. DOWNLOAD STATUS
# ------------------------------------------------

print("\n--- DOWNLOAD STATUS ---")

display(
    canonical_images["download_status"]
    .value_counts(dropna=False)
    .to_frame("records")
)

# ------------------------------------------------
# 3. IMAGE DIMENSIONS
# ------------------------------------------------

print("\n--- IMAGE DIMENSIONS ---")

print("Missing widths:", canonical_images["width"].isna().sum())
print("Missing heights:", canonical_images["height"].isna().sum())

invalid_dimensions = canonical_images[
    canonical_images["width"].le(0)
    | canonical_images["height"].le(0)
]

print("Invalid dimensions:", len(invalid_dimensions))

# ------------------------------------------------
# 4. HASH COVERAGE
# ------------------------------------------------

print("\n--- HASH COVERAGE ---")

for col in ["content_hash_sha256", "perceptual_hash_dhash"]:
    available = canonical_images[col].notna().sum()

    print(
        f"{col}: {available}/{len(canonical_images)} "
        f"({available / len(canonical_images) * 100:.2f}%)"
    )

# ------------------------------------------------
# 5. EXACT DUPLICATE HASHES
# ------------------------------------------------

print("\n--- SHA-256 DUPLICATE CANDIDATES ---")

sha = canonical_images[
    canonical_images["content_hash_sha256"].notna()
]

duplicate_sha = sha[
    sha.duplicated("content_hash_sha256", keep=False)
].sort_values("content_hash_sha256")

print("Records involved in repeated SHA-256 hashes:", len(duplicate_sha))
print(
    "Unique repeated SHA-256 values:",
    duplicate_sha["content_hash_sha256"].nunique()
)

display(
    duplicate_sha[
        [
            "image_id",
            "canonical_product_id",
            "marketplace",
            "content_hash_sha256"
        ]
    ].head(20)
)

# ------------------------------------------------
# 6. OCR COVERAGE
# ------------------------------------------------

print("\n--- OCR COVERAGE ---")

ocr_available = (
    canonical_images["ocr_text"].notna()
    & canonical_images["ocr_text"].astype(str).str.strip().ne("")
)

print("OCR records:", int(ocr_available.sum()))
print(
    "OCR coverage:",
    round(ocr_available.mean() * 100, 2),
    "%"
)

print("\nAUDIT CELL COMPLETED — NO FILES MODIFIED")

In [ ]:
# ============================================================
# EXACT DUPLICATE IMAGE CLUSTER INVESTIGATION
# ============================================================

# Find repeated SHA-256 hashes
sha_data = canonical_images[
    canonical_images["content_hash_sha256"].notna()
].copy()

duplicate_hashes = (
    sha_data.groupby("content_hash_sha256")
    .filter(lambda group: len(group) > 1)
)

# Join product metadata
product_info = canonical_products[
    ["canonical_product_id", "title", "brand", "category", "marketplace"]
].copy()

duplicate_details = duplicate_hashes.merge(
    product_info,
    on="canonical_product_id",
    how="left",
    suffixes=("_image", "_product"),
    validate="many_to_one"
)

columns_to_show = [
    "image_id",
    "canonical_product_id",
    "marketplace_image",
    "title",
    "brand",
    "category",
    "image_url",
    "local_path",
    "content_hash_sha256",
    "perceptual_hash_dhash"
]

display(duplicate_details[columns_to_show])

print("\nCLUSTER SUMMARY")

print(
    "Unique products:",
    duplicate_details["canonical_product_id"].nunique()
)

print(
    "Unique categories:",
    duplicate_details["category"].nunique()
)

print(
    "Unique brands:",
    duplicate_details["brand"].nunique()
)

print(
    "Unique image URLs:",
    duplicate_details["image_url"].nunique()
)

print(
    "Unique local paths:",
    duplicate_details["local_path"].nunique()
)


In [ ]:
# ============================================================
# DUPLICATE CLUSTER — DETAILED SOURCE INVESTIGATION
# ============================================================

duplicate_hash = (
    "fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144"
)

cluster = canonical_images[
    canonical_images["content_hash_sha256"]
    .astype("string")
    .str.startswith(duplicate_hash)
    .fillna(False)
].copy()

print("DUPLICATE RECORD COUNT:", len(cluster))

print("\n--- ORIGINAL IMAGE RECORDS ---")

display(
    cluster[[
        "image_id",
        "canonical_product_id",
        "marketplace",
        "image_url",
        "local_path",
        "content_hash_sha256",
        "perceptual_hash_dhash"
    ]].to_string(index=False)
)

# Join product metadata separately
metadata = canonical_products[[
    "canonical_product_id",
    "title",
    "brand",
    "category"
]].copy()

details = cluster.merge(
    metadata,
    on="canonical_product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\n--- PRODUCT METADATA MATCHING ---")

display(
    details[[
        "canonical_product_id",
        "title",
        "brand",
        "category",
        "_merge"
    ]].to_string(index=False)
)

print("\n--- CLUSTER STATISTICS ---")

print("Unique image URLs:", cluster["image_url"].nunique())
print("Unique local paths:", cluster["local_path"].nunique())
print("Unique marketplaces:", cluster["marketplace"].nunique())
print("Unique product IDs:", cluster["canonical_product_id"].nunique())
print("Unmatched metadata rows:", (details["_merge"] != "both").sum())

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

# Use the five-record duplicate cluster from the previous cell
verification = cluster.copy()

def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


results = []

for _, row in verification.iterrows():

    relative_path = str(row["local_path"]).replace("\\", "/")
    file_path = PROJECT_ROOT / relative_path

    exists = file_path.is_file()

    actual_hash = (
        calculate_sha256(file_path)
        if exists else None
    )

    results.append({
        "image_id": row["image_id"],
        "file_exists": exists,
        "file_size": file_path.stat().st_size if exists else None,
        "stored_hash": row["content_hash_sha256"],
        "actual_hash": actual_hash,
        "hash_matches": (
            actual_hash == row["content_hash_sha256"]
            if exists else False
        )
    })

verification_report = pd.DataFrame(results)

display(verification_report)

print("\nFILES FOUND:", verification_report["file_exists"].sum())
print("HASH MATCHES:", verification_report["hash_matches"].sum())

In [ ]:
# ============================================================
# PERCEPTUAL HASH (dHash) NEAR-DUPLICATE AUDIT
# ============================================================

from itertools import combinations

# Keep records with valid 64-bit hexadecimal dHash values
dhash_data = canonical_images[
    canonical_images["perceptual_hash_dhash"].notna()
].copy()

dhash_data["dhash_clean"] = (
    dhash_data["perceptual_hash_dhash"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_mask = dhash_data["dhash_clean"].str.fullmatch(r"[0-9a-f]{16}")
invalid_count = int((~valid_mask).sum())

dhash_data = dhash_data.loc[valid_mask].copy()

# Convert hexadecimal hashes to integers
dhash_data["dhash_int"] = dhash_data["dhash_clean"].apply(
    lambda x: int(x, 16)
)

print("VALID DHASH RECORDS:", len(dhash_data))
print("INVALID DHASH RECORDS:", invalid_count)

# ------------------------------------------------
# HAMMING DISTANCE
# ------------------------------------------------

def hamming_distance(hash1, hash2):
    return (hash1 ^ hash2).bit_count()


# ------------------------------------------------
# COMPARE EVERY UNIQUE PAIR
# ------------------------------------------------

records = dhash_data.to_dict("records")
pairs = []

for a, b in combinations(records, 2):

    distance = hamming_distance(
        a["dhash_int"],
        b["dhash_int"]
    )

    pairs.append({
        "image_id_1": a["image_id"],
        "image_id_2": b["image_id"],
        "product_id_1": a["canonical_product_id"],
        "product_id_2": b["canonical_product_id"],
        "marketplace_1": a["marketplace"],
        "marketplace_2": b["marketplace"],
        "sha256_1": a["content_hash_sha256"],
        "sha256_2": b["content_hash_sha256"],
        "dhash_1": a["dhash_clean"],
        "dhash_2": b["dhash_clean"],
        "hamming_distance": distance,
        "same_product": (
            a["canonical_product_id"]
            == b["canonical_product_id"]
        ),
        "same_sha256": (
            pd.notna(a["content_hash_sha256"])
            and pd.notna(b["content_hash_sha256"])
            and a["content_hash_sha256"]
            == b["content_hash_sha256"]
        )
    })

dhash_pairs = pd.DataFrame(pairs)

print("\nTOTAL UNIQUE IMAGE PAIRS:", len(dhash_pairs))

# ------------------------------------------------
# DISTANCE DISTRIBUTION
# ------------------------------------------------

print("\n--- HAMMING DISTANCE DISTRIBUTION ---")

display(
    dhash_pairs["hamming_distance"]
    .value_counts()
    .sort_index()
    .rename_axis("hamming_distance")
    .to_frame("pair_count")
)

# ------------------------------------------------
# CANDIDATE COUNTS AT MULTIPLE THRESHOLDS
# ------------------------------------------------

print("\n--- CANDIDATE COUNTS ---")

for threshold in [0, 2, 5, 10]:
    count = (
        dhash_pairs["hamming_distance"] <= threshold
    ).sum()

    print(f"Distance <= {threshold}: {count}")

# ------------------------------------------------
# INSPECT CLOSEST PAIRS
# ------------------------------------------------

print("\n--- 30 CLOSEST IMAGE PAIRS ---")

closest_pairs = dhash_pairs.sort_values(
    ["hamming_distance", "image_id_1"]
).head(30)

display(closest_pairs)

print("\nAUDIT COMPLETED — NO FILES MODIFIED")

In [ ]:
# Inspect all zero-distance dHash pairs
zero_pairs = dhash_pairs[
    dhash_pairs["hamming_distance"] == 0
].copy()

# Add product metadata for image 1
product_lookup = canonical_products[
    ["canonical_product_id", "title", "brand", "category"]
].drop_duplicates("canonical_product_id")

zero_pairs = zero_pairs.merge(
    product_lookup.add_prefix("p1_"),
    left_on="product_id_1",
    right_on="p1_canonical_product_id",
    how="left"
)

# Add product metadata for image 2
zero_pairs = zero_pairs.merge(
    product_lookup.add_prefix("p2_"),
    left_on="product_id_2",
    right_on="p2_canonical_product_id",
    how="left"
)

display(zero_pairs[[
    "image_id_1", "product_id_1", "p1_title", "p1_brand",
    "image_id_2", "product_id_2", "p2_title", "p2_brand",
    "same_product", "same_sha256", "sha256_1", "sha256_2"
]])

print("Zero-distance pairs:", len(zero_pairs))
print("Same SHA-256 pairs:", int(zero_pairs["same_sha256"].sum()))
print("Different SHA-256 pairs:", int((~zero_pairs["same_sha256"]).sum()))

In [ ]:
different_hash_pairs = zero_pairs[
    zero_pairs["same_sha256"] == False
].copy()

display(different_hash_pairs[[
    "image_id_1",
    "product_id_1",
    "p1_title",
    "p1_brand",
    "image_id_2",
    "product_id_2",
    "p2_title",
    "p2_brand",
    "sha256_1",
    "sha256_2",
    "dhash_1",
    "dhash_2"
]].T)

In [ ]:
for _, row in different_hash_pairs.iterrows():
    for image_id in [row["image_id_1"], row["image_id_2"]]:
        image_row = canonical_images[
            canonical_images["image_id"] == image_id
        ]

        print("\nImage ID:", image_id)
        print("Local path:", image_row["local_path"].iloc[0])
        print("Exists:", Path(image_row["local_path"].iloc[0]).exists())

In [ ]:
candidate_ids = [
    "amazon_B07QMZS32S_img_0",
    "amazon_B0BWZX4TQ1_img_0"
]

candidate_records = canonical_images[
    canonical_images["image_id"].isin(candidate_ids)
].copy()

display(candidate_records[[
    "image_id",
    "canonical_product_id",
    "image_url",
    "local_path",
    "image_type",
    "download_status",
    "file_size_bytes",
    "width",
    "height",
    "content_hash_sha256",
    "perceptual_hash_dhash"
]].T)

In [ ]:
from pathlib import Path
import hashlib

for _, row in candidate_records.iterrows():
    image_path = PROJECT_ROOT / row["local_path"]

    print("\nImage:", row["image_id"])
    print("Resolved path:", image_path)
    print("Exists:", image_path.exists())

    if image_path.exists():
        actual_size = image_path.stat().st_size

        with open(image_path, "rb") as f:
            actual_sha256 = hashlib.sha256(f.read()).hexdigest()

        print("Stored size:", row["file_size_bytes"])
        print("Actual size:", actual_size)
        print("Stored SHA:", row["content_hash_sha256"])
        print("Actual SHA:", actual_sha256)
        print("SHA matches:", actual_sha256 == row["content_hash_sha256"])

### Perceptual Hash Duplicate Audit

A pairwise Hamming-distance audit was performed on all 520 valid
perceptual hashes, resulting in 134,940 unique image pairs.

Key findings:

- 11 image pairs had a Hamming distance of zero.
- 10 pairs were confirmed as exact duplicates using matching
  SHA-256 hashes.
- One pair had identical dHash values but different SHA-256 hashes.
- The latter pair involved unrelated TRESemmé and Shiseido
  product listings. Both source files were verified against
  their stored SHA-256 hashes and file sizes.

Conclusion:

Exact duplicates were confirmed for 10 pairs. One additional
perceptual-hash collision candidate was identified.

Hamming-distance thresholds remain exploratory and are not
approved production thresholds. Near-duplicate candidates
require additional visual validation.

No source files or datasets were modified during this audit.

In [ ]:
# Inspect existing embedding-related variables
print("Embedding-related notebook variables:")

for name in sorted(globals()):
    if any(term in name.lower() for term in ["embed", "resnet", "feature_vector"]):
        obj = globals()[name]
        print(
            name,
            "| type:", type(obj).__name__,
            "| shape:", getattr(obj, "shape", None)
        )

In [ ]:
# Inspect existing CV artifacts without modifying anything
for folder in [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "models"
]:
    print(f"\n--- {folder.name} ---")
    for path in sorted(folder.glob("*")):
        if any(term in path.name.lower() for term in ["embed", "resnet", "vision", "image"]):
            print(path.name)

In [ ]:
print("Embedding-related notebook variables:")

for name in sorted(globals()):
    if any(term in name.lower() for term in [
        "embed", "resnet", "feature_vector"
    ]):
        obj = globals()[name]
        print(
            name,
            "| type:", type(obj).__name__,
            "| shape:", getattr(obj, "shape", None)
        )

In [ ]:
dense_images_path = (
    PROJECT_ROOT / "data" / "interim" / "dense_product_images.parquet"
)

dense_images = pd.read_parquet(dense_images_path)

print("Shape:", dense_images.shape)
print("\nColumns:")
print(dense_images.columns.tolist())

print("\nData types:")
print(dense_images.dtypes)

display(dense_images.head(3))

In [ ]:
# Search notebook code for embedding generation
import json

notebook_path = PROJECT_ROOT / "notebooks" / "05_computer_vision.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

keywords = ["resnet", "embedding", "pretrained", "avgpool", "torchvision"]

for i, cell in enumerate(notebook_data["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword in source.lower() for keyword in keywords):
        print(f"\n{'='*60}")
        print(f"CELL {i}")
        print(source[:3000])

In [ ]:
# Inspect saved notebook source around the ResNet implementation
for i in range(48, min(65, len(notebook_data["cells"]))):
    cell = notebook_data["cells"][i]

    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        print(f"\n{'='*70}")
        print(f"CELL {i}")
        print(source[:5000])

        # Show whether the cell has saved output
        outputs = cell.get("outputs", [])
        print("Saved outputs:", len(outputs))

In [ ]:
# Focused inspection of likely embedding objects
for name in [
    "model",
    "weights",
    "preprocess",
    "embedding_model",
    "image_embeddings",
    "embeddings",
    "embedding_vectors",
    "resnet_features",
    "features"
]:
    if name in globals():
        obj = globals()[name]
        print(
            name,
            "| type:", type(obj).__name__,
            "| shape:", getattr(obj, "shape", None)
        )

In [ ]:
for i in range(49, 56):
    cell = notebook_data["cells"][i]

    print(f"\n{'='*60}")
    print(f"CELL {i}")

    if cell["cell_type"] == "code":
        print("".join(cell["source"]))

        outputs = cell.get("outputs", [])
        print("\nSaved outputs:", len(outputs))

In [ ]:
source_49 = "".join(notebook_data["cells"][49]["source"])
lines_49 = source_49.splitlines()

print("Total lines:", len(lines_49))

for start in range(0, len(lines_49), 35):
    end = min(start + 35, len(lines_49))

    print(f"\n--- CELL 49: LINES {start+1}-{end} ---")
    for line_no in range(start, end):
        print(f"{line_no+1:03}: {lines_49[line_no]}")

In [ ]:
# Embedding extraction section
for i in range(42, 71):
    print(f"{i+1:03}: {lines_49[i]}")

In [ ]:
validation_path = (
    PROJECT_ROOT / "data" / "processed"
    / "vision_manual_evaluation.csv"
)

validation_df = pd.read_csv(validation_path)

print(validation_df.shape)
print(validation_df.columns.tolist())
display(validation_df)

In [ ]:

# ============================================================
# RESNET-18 EMBEDDING AUDIT — READ ONLY
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("RESNET-18 EMBEDDING AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. CHECK WHETHER REQUIRED VARIABLES EXIST
# ------------------------------------------------------------

required_variables = [
    "embedding_df",
    "embedding_matrix",
    "embeddings",
    "image_df",
    "failed_images",
    "valid_rows"
]

print("\n--- VARIABLE AVAILABILITY ---")

for name in required_variables:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}")

# Stop safely if extraction variables are missing
missing = [name for name in required_variables if name not in globals()]

if missing:
    print("\nAUDIT CANNOT CONTINUE.")
    print("Missing variables:", missing)
    print("\nCell 49 must be executed successfully before this audit.")
else:

    # --------------------------------------------------------
    # 2. BASIC EMBEDDING STATISTICS
    # --------------------------------------------------------

    print("\n--- EMBEDDING STATISTICS ---")

    print("Original downloaded image records:", len(image_df))
    print("Successfully embedded images:", len(embedding_df))
    print("Failed images:", len(failed_images))
    print("Embedding matrix shape:", embedding_matrix.shape)

    # --------------------------------------------------------
    # 3. IMAGE ID INTEGRITY
    # --------------------------------------------------------

    print("\n--- IMAGE ID INTEGRITY ---")

    print(
        "Unique image IDs:",
        embedding_df["image_id"].nunique()
    )

    print(
        "Duplicate image IDs:",
        embedding_df["image_id"].duplicated().sum()
    )

    print(
        "Missing image IDs:",
        embedding_df["image_id"].isna().sum()
    )

    print(
        "Missing product IDs:",
        embedding_df["canonical_product_id"].isna().sum()
    )

    # --------------------------------------------------------
    # 4. EMBEDDING DIMENSION VALIDATION
    # --------------------------------------------------------

    print("\n--- DIMENSION VALIDATION ---")

    print("Expected dimensions per image: 512")
    print(
        "Actual dimensions per image:",
        embedding_matrix.shape[1]
        if embedding_matrix.ndim == 2 else "INVALID"
    )

    print(
        "Number of embedding rows:",
        embedding_matrix.shape[0]
    )

    print(
        "Embedding/DataFrame row alignment:",
        embedding_matrix.shape[0] == len(embedding_df)
    )

    # --------------------------------------------------------
    # 5. NUMERICAL INTEGRITY
    # --------------------------------------------------------

    print("\n--- NUMERICAL INTEGRITY ---")

    print("Contains NaN:", np.isnan(embedding_matrix).any())
    print("Contains infinity:", np.isinf(embedding_matrix).any())

    if embedding_matrix.size > 0:

        norms = np.linalg.norm(embedding_matrix, axis=1)

        print("Minimum vector norm:", norms.min())
        print("Maximum vector norm:", norms.max())

        print(
            "All vectors approximately L2 normalized:",
            np.allclose(norms, 1.0, atol=1e-5)
        )

    # --------------------------------------------------------
    # 6. FAILED IMAGE INSPECTION
    # --------------------------------------------------------

    print("\n--- FAILED IMAGE INSPECTION ---")

    if failed_images:
        failed_df = pd.DataFrame(failed_images)
        print("Failed image count:", len(failed_df))
        display(failed_df.head(10))
    else:
        print("No failed images recorded.")

    # --------------------------------------------------------
    # 7. IMAGE-TO-PRODUCT MAPPING
    # --------------------------------------------------------

    print("\n--- IMAGE / PRODUCT MAPPING ---")

    display(
        embedding_df[
            ["image_id", "canonical_product_id"]
        ].head(10)
    )

    # --------------------------------------------------------
    # 8. EMBEDDING PERSISTENCE CHECK
    # --------------------------------------------------------

    print("\n--- EXISTING EMBEDDING FILES ---")

    artifact_folders = [
        PROJECT_ROOT / "data" / "processed",
        PROJECT_ROOT / "data" / "interim",
        PROJECT_ROOT / "models"
    ]

    for folder in artifact_folders:
        print(f"\n{folder}")

        for path in sorted(folder.glob("*")):
            if any(
                term in path.name.lower()
                for term in ["embedding", "resnet", "vision_feature"]
            ):
                print(path.name)

    print("\n" + "=" * 65)
    print("RESNET-18 AUDIT COMPLETED")
    print("No files were modified by this audit.")
    print("=" * 65)


In [ ]:
print("Model available:", "model" in globals())
print("Preprocessing available:", "preprocess" in globals())

image_check = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)

downloaded_check = image_check[
    image_check["download_status"].eq("DOWNLOADED")
    & image_check["local_path"].notna()
].copy()

print("Total canonical image records:", len(image_check))
print("Downloaded image records:", len(downloaded_check))
print(
    "Existing local files:",
    sum(
        (PROJECT_ROOT / str(p)).exists()
        for p in downloaded_check["local_path"]
    )
)

In [ ]:

import torch
import torchvision
from torchvision import models
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Remove classification layer
model.fc = torch.nn.Identity()

model.eval()

# CPU inference, as in the original notebook
device = torch.device("cpu")
model = model.to(device)

# Preprocessing associated with pretrained weights
preprocess = weights.transforms()

print("\nResNet-18 loaded successfully!")
print("Device:", device)
print("Embedding dimensions:", 512)
print("Preprocessing loaded successfully!")


In [ ]:
print("Model available:", "model" in globals())
print("Preprocessing available:", "preprocess" in globals())
print("Device:", next(model.parameters()).device)

In [ ]:

# ============================================================
# RESNET-18 IMAGE EMBEDDING EXTRACTION
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from PIL import Image
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD DOWNLOADED IMAGE RECORDS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)

image_df = pd.read_parquet(MANIFEST_PATH)

image_df = (
    image_df[
        image_df["download_status"].eq("DOWNLOADED")
        & image_df["local_path"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

print("Downloaded image records:", len(image_df))

# ------------------------------------------------------------
# 2. PREPARE EXTRACTION
# ------------------------------------------------------------

model.eval()
device = torch.device("cpu")
model = model.to(device)

embeddings = []
valid_rows = []
failed_images = []

# ------------------------------------------------------------
# 3. EXTRACT EMBEDDINGS
# ------------------------------------------------------------

with torch.inference_mode():

    for idx, row in image_df.iterrows():

        image_path = PROJECT_ROOT / str(row["local_path"])

        try:
            with Image.open(image_path) as img:
                image = img.convert("RGB")
                tensor = preprocess(image).unsqueeze(0).to(device)

            vector = model(tensor)

            vector = F.normalize(
                vector,
                p=2,
                dim=1
            )

            embeddings.append(
                vector.squeeze(0).cpu().numpy()
            )

            valid_rows.append(idx)

        except Exception as exc:
            failed_images.append({
                "image_id": row["image_id"],
                "error": str(exc)
            })

        if (idx + 1) % 50 == 0:
            print(
                f"Processed {idx + 1}/{len(image_df)} images"
            )

# ------------------------------------------------------------
# 4. BUILD EMBEDDING DATAFRAME
# ------------------------------------------------------------

embedding_df = (
    image_df.iloc[valid_rows]
    .reset_index(drop=True)
)

embedding_matrix = (
    np.stack(embeddings).astype(np.float32)
    if embeddings
    else np.empty((0, 512), dtype=np.float32)
)

print("\n--- EMBEDDING RESULTS ---")
print("Images processed:", len(embedding_df))
print("Failed images:", len(failed_images))
print("Embedding matrix shape:", embedding_matrix.shape)


In [ ]:

# ============================================================
# RESNET-18 EMBEDDING VALIDATION
# ============================================================

print("Embedding dataframe shape:", embedding_df.shape)
print("Embedding matrix shape:", embedding_matrix.shape)

print("\n--- DIMENSION CHECK ---")
print("Expected shape:", (520, 512))
print("Actual shape:", embedding_matrix.shape)
print("Correct dimensions:", embedding_matrix.shape == (520, 512))

print("\n--- IMAGE ID CHECK ---")
print("Unique image IDs:", embedding_df["image_id"].nunique())
print("Duplicate image IDs:", embedding_df["image_id"].duplicated().sum())
print("Missing image IDs:", embedding_df["image_id"].isna().sum())

print("\n--- NUMERICAL CHECK ---")
print("Contains NaN:", np.isnan(embedding_matrix).any())
print("Contains infinity:", np.isinf(embedding_matrix).any())

norms = np.linalg.norm(embedding_matrix, axis=1)

print("Minimum vector norm:", norms.min())
print("Maximum vector norm:", norms.max())
print(
    "All vectors L2 normalized:",
    np.allclose(norms, 1.0, atol=1e-5)
)

print("\n--- PRODUCT MAPPING ---")
print("Unique products represented:",
      embedding_df["canonical_product_id"].nunique())

display(
    embedding_df[
        ["image_id", "canonical_product_id"]
    ].head()
)


In [ ]:

# ============================================================
# SAVE RESNET-18 EMBEDDINGS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

embedding_path = OUTPUT_DIR / "image_embeddings_resnet18.npy"
metadata_path = OUTPUT_DIR / "image_embeddings_resnet18_metadata.parquet"

# Safety: never overwrite existing files
if embedding_path.exists() or metadata_path.exists():
    raise FileExistsError(
        "An embedding artifact already exists. "
        "Inspect it before deciding whether to replace anything."
    )

# Keep only the identifiers needed to map vectors to source images
embedding_metadata = embedding_df[
    ["image_id", "canonical_product_id", "marketplace"]
].copy()

# Validate row alignment before saving
assert len(embedding_metadata) == len(embedding_matrix)
assert embedding_metadata["image_id"].is_unique
assert embedding_matrix.shape == (520, 512)

# Save both artifacts
np.save(embedding_path, embedding_matrix)

embedding_metadata.to_parquet(
    metadata_path,
    index=False
)

print("Embedding artifacts saved successfully.")

print("\nMatrix:", embedding_path)
print("Metadata:", metadata_path)

print("\nMatrix shape:", embedding_matrix.shape)
print("Metadata shape:", embedding_metadata.shape)


In [ ]:
saved_matrix = np.load(embedding_path)
saved_metadata = pd.read_parquet(metadata_path)

print("Saved matrix shape:", saved_matrix.shape)
print("Saved metadata shape:", saved_metadata.shape)

print(
    "Matrix identical:",
    np.array_equal(saved_matrix, embedding_matrix)
)

print(
    "Metadata IDs aligned:",
    saved_metadata["image_id"].equals(embedding_metadata["image_id"])
)

In [ ]:

# ============================================================
# MANUAL EVALUATION SHEET — READ-ONLY AUDIT
# ============================================================

EVALUATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vision_manual_evaluation.csv"
)

evaluation_df = pd.read_csv(EVALUATION_PATH)

print("Shape:", evaluation_df.shape)
print("\nColumns:")
print(evaluation_df.columns.tolist())

print("\n--- MANUAL LABEL DISTRIBUTION ---")
print(
    evaluation_df["manual_label"]
    .fillna("NOT_ANNOTATED")
    .value_counts(dropna=False)
)

print("\n--- DISTANCE SUMMARY ---")
print(evaluation_df["dhash_distance"].describe())

print("\n--- COSINE SIMILARITY SUMMARY ---")
print(evaluation_df["cosine_similarity"].describe())

print("\n--- EXACT HASH MATCHES ---")
print(evaluation_df["same_sha256"].value_counts(dropna=False))

print("\n--- PAIR REVIEW ---")
display(
    evaluation_df[
        [
            "image_a",
            "image_b",
            "dhash_distance",
            "cosine_similarity",
            "same_sha256",
            "manual_label",
            "notes"
        ]
    ].sort_values(
        by=["dhash_distance", "cosine_similarity"],
        ascending=[True, False]
    ).head(20)
)


In [ ]:

# ============================================================
# SELECT TOP 10 PAIRS FOR MANUAL VISUAL INSPECTION
# READ-ONLY — DOES NOT CHANGE ORIGINAL LABELS
# ============================================================

inspection_df = evaluation_df.copy()

inspection_df["manual_label"] = (
    inspection_df["manual_label"].fillna("NOT_ANNOTATED")
)

inspection_df = inspection_df.sort_values(
    by=["dhash_distance", "cosine_similarity"],
    ascending=[True, False]
).head(10).reset_index(drop=True)

display(
    inspection_df[
        [
            "image_a",
            "image_b",
            "product_a",
            "product_b",
            "dhash_distance",
            "cosine_similarity",
            "manual_label"
        ]
    ]
)


In [ ]:

# ============================================================
# VISUAL INSPECTION — TOP 10 IMAGE PAIRS
# READ-ONLY
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Reload source image metadata if needed
if "image_df" not in globals():
    image_df = pd.read_parquet(
        PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
    )

image_lookup = (
    image_df.drop_duplicates("image_id")
    .set_index("image_id")
)

fig, axes = plt.subplots(
    nrows=len(inspection_df),
    ncols=2,
    figsize=(12, 4 * len(inspection_df))
)

for idx, pair in inspection_df.iterrows():
    for col, image_id in enumerate([pair["image_a"], pair["image_b"]]):
        ax = axes[idx, col]

        if image_id not in image_lookup.index:
            ax.text(0.5, 0.5, "Image ID not found", ha="center")
            ax.axis("off")
            continue

        image_record = image_lookup.loc[image_id]
        image_path = PROJECT_ROOT / str(image_record["local_path"])

        try:
            with Image.open(image_path) as img:
                ax.imshow(img.convert("RGB"))
            ax.set_title(
                f"{image_id}\n"
                f"Product: {image_record['canonical_product_id']}",
                fontsize=8
            )
        except Exception as exc:
            ax.text(0.5, 0.5, f"Could not load image:\n{exc}", ha="center")

        ax.axis("off")

    fig.text(
        0.5,
        1 - (idx + 0.5) / len(inspection_df),
        (
            f"Pair {idx + 1} | "
            f"dHash: {pair['dhash_distance']} | "
            f"Cosine: {pair['cosine_similarity']:.3f} | "
            f"Label: {pair['manual_label']}"
        ),
        ha="center",
        fontsize=10
    )

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# COMPUTER VISION — MANUAL VISUAL EVALUATION
# TOP 10 IMAGE PAIRS
#
# Purpose:
# 1. Store manually reviewed visual labels.
# 2. Preserve original evaluation data.
# 3. Save annotations as a separate CSV.
# 4. Prevent accidental overwriting.
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. DEFINE INPUT AND OUTPUT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

EVALUATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vision_manual_evaluation.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vision_manual_visual_review.csv"
)

# ------------------------------------------------------------
# 2. LOAD ORIGINAL EVALUATION DATA
# ------------------------------------------------------------

evaluation_df = pd.read_csv(EVALUATION_PATH)

print("Original evaluation shape:", evaluation_df.shape)

# Select the same top 10 pairs for inspection
inspection_df = (
    evaluation_df.copy()
    .sort_values(
        by=["dhash_distance", "cosine_similarity"],
        ascending=[True, False]
    )
    .head(10)
    .reset_index(drop=True)
)

print("Pairs selected for review:", len(inspection_df))

# ------------------------------------------------------------
# 3. RECORD MANUAL VISUAL LABELS
# ------------------------------------------------------------

# Labels are based on the uploaded contact sheets.
# Pair 1: similar-looking bottles, visibly different products.
# Pairs 2-10: visibly different product categories.

reviewed_labels = {
    0: (
        "different_products",
        "Similar bottle shape, but visibly different colour and label."
    ),
    1: (
        "different_products",
        "Costume product and children's tricycle."
    ),
    2: (
        "different_products",
        "Cosmetic serum and toy stroller."
    ),
    3: (
        "different_products",
        "Decorative flowers and children's tricycle."
    ),
    4: (
        "different_products",
        "Dress and monster truck."
    ),
    5: (
        "different_products",
        "Hair styling gel and clothing."
    ),
    6: (
        "different_products",
        "Knife block and dollhouse."
    ),
    7: (
        "different_products",
        "Jacket and toy stroller."
    ),
    8: (
        "different_products",
        "Plush toy and resin container."
    ),
    9: (
        "different_products",
        "Pink cosmetic/bath product and Android car stereo."
    )
}

# ------------------------------------------------------------
# 4. CREATE SEPARATE ANNOTATION COLUMNS
# ------------------------------------------------------------

manual_review_df = inspection_df.copy()

manual_review_df["visual_label"] = "NOT_REVIEWED"
manual_review_df["visual_notes"] = None

for idx, (label, note) in reviewed_labels.items():
    manual_review_df.loc[idx, "visual_label"] = label
    manual_review_df.loc[idx, "visual_notes"] = note

# ------------------------------------------------------------
# 5. VALIDATE ANNOTATIONS
# ------------------------------------------------------------

assert len(manual_review_df) == 10

assert manual_review_df["visual_label"].notna().all()

assert (
    manual_review_df["visual_label"]
    .isin([
        "same_image",
        "same_product",
        "different_products",
        "uncertain",
        "NOT_REVIEWED"
    ])
    .all()
)

assert manual_review_df["image_a"].notna().all()
assert manual_review_df["image_b"].notna().all()

print("\n--- VALIDATION ---")
print("Total reviewed pairs:", len(manual_review_df))
print("Missing visual labels:", manual_review_df["visual_label"].isna().sum())

# ------------------------------------------------------------
# 6. SAVE AS A NEW FILE — NEVER OVERWRITE
# ------------------------------------------------------------

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output file already exists:\n{OUTPUT_PATH}\n"
        "Inspect the existing file before deciding what to do."
    )

manual_review_df.to_csv(OUTPUT_PATH, index=False)

print("\nManual visual evaluation saved successfully!")
print("Output:", OUTPUT_PATH)

# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n--- VISUAL LABEL DISTRIBUTION ---")

print(
    manual_review_df["visual_label"]
    .value_counts(dropna=False)
)

print("\n--- FINAL REVIEW TABLE ---")

display(
    manual_review_df[
        [
            "image_a",
            "image_b",
            "dhash_distance",
            "cosine_similarity",
            "same_sha256",
            "visual_label",
            "visual_notes"
        ]
    ]
)

# ------------------------------------------------------------
# 8. CONFIRM ORIGINAL DATASET WAS NOT MODIFIED
# ------------------------------------------------------------

print("\nOriginal source file still exists:", EVALUATION_PATH.exists())
print("Separate annotation file exists:", OUTPUT_PATH.exists())


In [ ]:

from pathlib import Path

processed_dir = PROJECT_ROOT / "data" / "processed"

print("--- ANNOTATION FILE CHECK ---")

for path in processed_dir.glob("vision_manual_visual_review*"):
    print("Path:", path)
    print("Is file:", path.is_file())
    print("Is directory:", path.is_dir())

print("\n--- EXPECTED FILE ---")

expected_path = processed_dir / "vision_manual_visual_review.csv"

print("Expected path:", expected_path)
print("Exists:", expected_path.is_file())

if expected_path.is_file():
    check_df = pd.read_csv(expected_path)
    print("Shape:", check_df.shape)
    print("Unique image pairs:", len(
        check_df[["image_a", "image_b"]].drop_duplicates()
    ))
    print(check_df["visual_label"].value_counts())


In [ ]:

# ============================================================
# PHASE 2 — PRODUCT-LEVEL CV FEATURE ENGINEERING
# STEP 1: SOURCE ARTIFACT AUDIT
# READ-ONLY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# ------------------------------------------------------------
# 1. LOAD SOURCE ARTIFACTS
# ------------------------------------------------------------

products = pd.read_parquet(
    PROCESSED_DIR / "canonical_products.parquet"
)

images = pd.read_parquet(
    PROCESSED_DIR / "canonical_images.parquet"
)

embeddings = np.load(
    PROCESSED_DIR / "image_embeddings_resnet18.npy"
)

embedding_metadata = pd.read_parquet(
    PROCESSED_DIR / "image_embeddings_resnet18_metadata.parquet"
)

visual_review = pd.read_csv(
    PROCESSED_DIR / "vision_manual_visual_review.csv"
)

# ------------------------------------------------------------
# 2. BASIC SHAPE AUDIT
# ------------------------------------------------------------

print("=" * 60)
print("SOURCE ARTIFACT SHAPES")
print("=" * 60)

print("Canonical products:", products.shape)
print("Canonical images:", images.shape)
print("Embedding matrix:", embeddings.shape)
print("Embedding metadata:", embedding_metadata.shape)
print("Manual visual review:", visual_review.shape)

# ------------------------------------------------------------
# 3. CANONICAL PRODUCT KEY AUDIT
# ------------------------------------------------------------

print("\n--- PRODUCT KEY AUDIT ---")

print("Product ID column exists:",
      "canonical_product_id" in products.columns)

print("Unique canonical products:",
      products["canonical_product_id"].nunique())

print("Duplicate product IDs:",
      products["canonical_product_id"].duplicated().sum())

print("Missing product IDs:",
      products["canonical_product_id"].isna().sum())

# ------------------------------------------------------------
# 4. IMAGE DATA SCHEMA
# ------------------------------------------------------------

print("\n--- IMAGE COLUMNS ---")
print(images.columns.tolist())

print("\n--- IMAGE DOWNLOAD STATUS ---")

if "download_status" in images.columns:
    print(images["download_status"].value_counts(dropna=False))

print("\n--- IMAGE COVERAGE ---")

print(
    "Unique products with image records:",
    images["canonical_product_id"].nunique()
)

print(
    "Unique image IDs:",
    images["image_id"].nunique()
)

print(
    "Duplicate image IDs:",
    images["image_id"].duplicated().sum()
)

# ------------------------------------------------------------
# 5. IMAGE EVIDENCE AVAILABILITY
# ------------------------------------------------------------

evidence_columns = [
    "local_path",
    "file_size_bytes",
    "width",
    "height",
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
    "ocr_packaging_match_score"
]

print("\n--- IMAGE EVIDENCE COVERAGE ---")

for col in evidence_columns:
    if col in images.columns:
        print(
            f"{col}: "
            f"{images[col].notna().sum()} / {len(images)}"
        )
    else:
        print(f"{col}: COLUMN NOT FOUND")

# ------------------------------------------------------------
# 6. EMBEDDING MAPPING AUDIT
# ------------------------------------------------------------

print("\n--- RESNET EMBEDDING AUDIT ---")

print("Embedding dimensions:", embeddings.shape)

print(
    "Metadata image IDs unique:",
    embedding_metadata["image_id"].is_unique
)

print(
    "Embedding image IDs missing from canonical images:",
    (~embedding_metadata["image_id"].isin(images["image_id"])).sum()
)

print(
    "Embedding products represented:",
    embedding_metadata["canonical_product_id"].nunique()
)

# ------------------------------------------------------------
# 7. MANUAL VISUAL REVIEW AUDIT
# ------------------------------------------------------------

print("\n--- MANUAL VISUAL REVIEW ---")

print(visual_review["visual_label"].value_counts(dropna=False))

print(
    "Unique reviewed pairs:",
    visual_review[["image_a", "image_b"]].drop_duplicates().shape[0]
)

# ------------------------------------------------------------
# 8. PRODUCT-LEVEL COVERAGE SUMMARY
# ------------------------------------------------------------

print("\n--- PRODUCT UNIVERSE ---")

print("Total canonical products:", len(products))

print(
    "Products with image records:",
    images["canonical_product_id"].nunique()
)

print(
    "Products represented in embeddings:",
    embedding_metadata["canonical_product_id"].nunique()
)

print("\nSOURCE AUDIT COMPLETED — NO FILES MODIFIED.")


In [ ]:

# ============================================================
# PHASE 2 — DETAILED CV SOURCE COVERAGE AUDIT
# READ-ONLY
# ============================================================

print("=" * 60)
print("IMAGE EVIDENCE COVERAGE")
print("=" * 60)

for col in [
    "local_path",
    "file_size_bytes",
    "width",
    "height",
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
    "ocr_packaging_match_score"
]:
    if col in images.columns:
        count = images[col].notna().sum()
        print(f"{col}: {count}/{len(images)} ({count / len(images) * 100:.2f}%)")

print("\n" + "=" * 60)
print("EMBEDDING MAPPING VALIDATION")
print("=" * 60)

print("Embedding matrix shape:", embeddings.shape)
print("Metadata shape:", embedding_metadata.shape)

print(
    "Metadata rows match matrix:",
    len(embedding_metadata) == len(embeddings)
)

print(
    "Unique embedding image IDs:",
    embedding_metadata["image_id"].nunique()
)

print(
    "Embedding image IDs missing from canonical images:",
    (~embedding_metadata["image_id"].isin(images["image_id"])).sum()
)

print(
    "Embedding product IDs missing from canonical products:",
    (~embedding_metadata["canonical_product_id"].isin(
        products["canonical_product_id"]
    )).sum()
)

print(
    "Unique products represented:",
    embedding_metadata["canonical_product_id"].nunique()
)

print("\n" + "=" * 60)
print("MANUAL VISUAL REVIEW")
print("=" * 60)

print("Rows:", len(visual_review))
print("Labels:")
print(visual_review["visual_label"].value_counts(dropna=False))

print("\n" + "=" * 60)
print("IMAGE STATUS BY PRODUCT")
print("=" * 60)

product_image_counts = (
    images.groupby("canonical_product_id")
    .agg(
        image_records=("image_id", "count"),
        downloaded=("download_status",
                    lambda x: (x == "DOWNLOADED").sum()),
        pending=("download_status",
                 lambda x: (x == "PENDING").sum())
    )
)

print("Products with image records:", len(product_image_counts))
print("Products with downloaded images:",
      (product_image_counts["downloaded"] > 0).sum())
print("Products with only pending images:",
      (
          (product_image_counts["downloaded"] == 0)
          & (product_image_counts["pending"] > 0)
      ).sum())

print("\nAUDIT COMPLETE — NO FILES MODIFIED.")


In [ ]:

# ============================================================
# PHASE 2 — PRODUCT-LEVEL CV FEATURE ENGINEERING
# STEP 2: IMAGE-LEVEL AGGREGATION
# READ-ONLY SOURCE DATA
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PREPARE IMAGE DATA
# ------------------------------------------------------------

image_work = images.copy()

# Explicit boolean indicators
image_work["is_downloaded"] = (
    image_work["download_status"].eq("DOWNLOADED")
)

image_work["is_pending"] = (
    image_work["download_status"].eq("PENDING")
)

image_work["has_local_path"] = (
    image_work["local_path"].notna()
)

image_work["has_sha256"] = (
    image_work["content_hash_sha256"].notna()
)

image_work["has_dhash"] = (
    image_work["perceptual_hash_dhash"].notna()
)

image_work["has_ocr"] = (
    image_work["ocr_text"].notna()
    & image_work["ocr_text"].astype("string").str.strip().ne("").fillna(False)
)

image_work["has_packaging_score"] = (
    image_work["packaging_consistency_score"].notna()
)

# ------------------------------------------------------------
# 2. AGGREGATE IMAGE COUNTS PER PRODUCT
# ------------------------------------------------------------

product_cv_features = (
    image_work.groupby("canonical_product_id")
    .agg(
        image_count_total=("image_id", "count"),
        image_count_downloaded=("is_downloaded", "sum"),
        image_count_pending=("is_pending", "sum"),
        image_count_with_local_path=("has_local_path", "sum"),
        image_count_with_sha256=("has_sha256", "sum"),
        image_count_with_dhash=("has_dhash", "sum"),
        image_count_with_ocr=("has_ocr", "sum"),
        image_count_with_packaging_score=(
            "has_packaging_score", "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. AGGREGATE HASH EVIDENCE
# ------------------------------------------------------------

hash_features = (
    image_work.dropna(subset=["content_hash_sha256"])
    .groupby("canonical_product_id")
    .agg(
        unique_sha256_count=(
            "content_hash_sha256", "nunique"
        )
    )
    .reset_index()
)

product_cv_features = product_cv_features.merge(
    hash_features,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 4. AGGREGATE PACKAGING SCORES
# ------------------------------------------------------------

packaging_features = (
    image_work.dropna(
        subset=["packaging_consistency_score"]
    )
    .groupby("canonical_product_id")
    .agg(
        packaging_score_mean=(
            "packaging_consistency_score", "mean"
        ),
        packaging_score_median=(
            "packaging_consistency_score", "median"
        ),
        packaging_score_max=(
            "packaging_consistency_score", "max"
        )
    )
    .reset_index()
)

product_cv_features = product_cv_features.merge(
    packaging_features,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 5. AGGREGATE EMBEDDING COVERAGE
# ------------------------------------------------------------

embedding_product_features = (
    embedding_metadata.groupby("canonical_product_id")
    .agg(
        embedding_image_count=("image_id", "nunique")
    )
    .reset_index()
)

product_cv_features = product_cv_features.merge(
    embedding_product_features,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 6. ADD EXPLICIT COVERAGE STATUS
# ------------------------------------------------------------

product_cv_features["image_evidence_status"] = np.select(
    [
        product_cv_features["image_count_downloaded"].gt(0),
        product_cv_features["image_count_pending"].gt(0)
    ],
    [
        "DOWNLOADED_IMAGE_EVIDENCE",
        "PENDING_IMAGE_ONLY"
    ],
    default="NO_IMAGE_RECORDS"
)

# ------------------------------------------------------------
# 7. ADD COVERAGE RATIOS
# ------------------------------------------------------------

product_cv_features["image_download_coverage"] = (
    product_cv_features["image_count_downloaded"]
    / product_cv_features["image_count_total"]
)

product_cv_features["ocr_coverage"] = (
    product_cv_features["image_count_with_ocr"]
    / product_cv_features["image_count_downloaded"]
).where(
    product_cv_features["image_count_downloaded"].gt(0)
)

product_cv_features["embedding_coverage"] = (
    product_cv_features["embedding_image_count"]
    / product_cv_features["image_count_downloaded"]
).where(
    product_cv_features["image_count_downloaded"].gt(0)
)

# ------------------------------------------------------------
# 8. VALIDATE IMAGE-LEVEL AGGREGATION
# ------------------------------------------------------------

print("--- IMAGE-LEVEL PRODUCT FEATURES ---")
print("Shape:", product_cv_features.shape)

print(
    "Unique product IDs:",
    product_cv_features["canonical_product_id"].nunique()
)

print(
    "Duplicate product IDs:",
    product_cv_features["canonical_product_id"].duplicated().sum()
)

print("\n--- EVIDENCE STATUS ---")
print(
    product_cv_features["image_evidence_status"]
    .value_counts(dropna=False)
)

print("\n--- SAMPLE FEATURES ---")
display(product_cv_features.head(10))

print("\nIMAGE-LEVEL AGGREGATION COMPLETE.")
print("No files saved or modified.")


In [ ]:

# ============================================================
# PHASE 2 — CROSS-PRODUCT EXACT IMAGE DUPLICATE FEATURES
# STEP 3
# READ-ONLY SOURCE DATA
# ============================================================

# 1. Select records with valid SHA-256 hashes
hash_df = images[
    images["content_hash_sha256"].notna()
].copy()

# 2. Find hashes appearing across multiple distinct products
hash_group_stats = (
    hash_df.groupby("content_hash_sha256")
    .agg(
        image_record_count=("image_id", "nunique"),
        distinct_product_count=("canonical_product_id", "nunique")
    )
    .reset_index()
)

cross_product_hashes = hash_group_stats[
    hash_group_stats["distinct_product_count"] > 1
].copy()

print("Unique SHA-256 hashes:", len(hash_group_stats))
print("Cross-product duplicate hash groups:",
      len(cross_product_hashes))

# 3. Identify image records belonging to those groups
cross_product_records = hash_df.merge(
    cross_product_hashes[["content_hash_sha256"]],
    on="content_hash_sha256",
    how="inner",
    validate="many_to_one"
)

# 4. Aggregate cross-product duplicate evidence per product
cross_product_duplicate_features = (
    cross_product_records.groupby("canonical_product_id")
    .agg(
        cross_product_duplicate_image_count=("image_id", "nunique"),
        cross_product_duplicate_hash_count=(
            "content_hash_sha256", "nunique"
        )
    )
    .reset_index()
)

# 5. Count distinct OTHER products sharing each hash
product_hash_pairs = (
    cross_product_records[
        ["canonical_product_id", "content_hash_sha256"]
    ]
    .drop_duplicates()
    .merge(
        cross_product_records[
            ["canonical_product_id", "content_hash_sha256"]
        ].drop_duplicates(),
        on="content_hash_sha256",
        suffixes=("_a", "_b")
    )
)

product_hash_pairs = product_hash_pairs[
    product_hash_pairs["canonical_product_id_a"]
    != product_hash_pairs["canonical_product_id_b"]
]

other_product_counts = (
    product_hash_pairs.groupby("canonical_product_id_a")
    ["canonical_product_id_b"]
    .nunique()
    .reset_index(name="distinct_other_products_sharing_image")
    .rename(
        columns={
            "canonical_product_id_a": "canonical_product_id"
        }
    )
)

# 6. Merge duplicate evidence into existing intermediate features
product_cv_features = product_cv_features.merge(
    cross_product_duplicate_features,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

product_cv_features = product_cv_features.merge(
    other_product_counts,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

# 7. Fill missing duplicate counts with zero:
# No matching cross-product duplicate evidence was found.
duplicate_columns = [
    "cross_product_duplicate_image_count",
    "cross_product_duplicate_hash_count",
    "distinct_other_products_sharing_image"
]

product_cv_features[duplicate_columns] = (
    product_cv_features[duplicate_columns]
    .fillna(0)
    .astype("int64")
)

# 8. Validate
print("\n--- CROSS-PRODUCT DUPLICATE SUMMARY ---")

print(
    "Products with cross-product duplicate images:",
    (
        product_cv_features[
            "cross_product_duplicate_image_count"
        ] > 0
    ).sum()
)

print(
    "Total affected image records:",
    product_cv_features[
        "cross_product_duplicate_image_count"
    ].sum()
)

print(
    "Total affected product-hash relationships:",
    product_cv_features[
        "cross_product_duplicate_hash_count"
    ].sum()
)

print("\n--- AFFECTED PRODUCTS ---")

display(
    product_cv_features[
        product_cv_features[
            "cross_product_duplicate_image_count"
        ] > 0
    ][
        [
            "canonical_product_id",
            "cross_product_duplicate_image_count",
            "cross_product_duplicate_hash_count",
            "distinct_other_products_sharing_image"
        ]
    ]
)

print("\n--- FINAL STRUCTURE ---")
print("Shape:", product_cv_features.shape)
print(
    "Duplicate product IDs:",
    product_cv_features["canonical_product_id"].duplicated().sum()
)

print("\nCROSS-PRODUCT DUPLICATE FEATURES COMPLETE.")
print("No files saved or modified.")


In [ ]:

# ============================================================
# PHASE 2 — PRODUCT-LEVEL RESNET EMBEDDING AGGREGATION
# STEP 4
# IN-MEMORY ONLY — NO FILES SAVED
# ============================================================

import numpy as np
import pandas as pd

# 1. Verify image-level embedding alignment
assert embedding_matrix.shape[0] == len(embedding_metadata)
assert embedding_matrix.shape[1] == 512

assert embedding_metadata["image_id"].is_unique
assert embedding_metadata["canonical_product_id"].notna().all()

# 2. Build image-to-product embedding mapping
embedding_rows = embedding_metadata[
    ["image_id", "canonical_product_id"]
].copy()

embedding_rows["embedding_index"] = np.arange(
    len(embedding_rows)
)

# 3. Aggregate image embeddings by product
product_embedding_records = []

for product_id, group in embedding_rows.groupby(
    "canonical_product_id",
    sort=False
):
    indices = group["embedding_index"].to_numpy()

    # Mean pooling across all available image embeddings
    mean_embedding = embedding_matrix[indices].mean(axis=0)

    # L2-normalize the pooled vector
    norm = np.linalg.norm(mean_embedding)

    if norm > 0:
        mean_embedding = mean_embedding / norm

    product_embedding_records.append({
        "canonical_product_id": product_id,
        "product_embedding": mean_embedding.astype(np.float32),
        "embedding_image_count": len(indices)
    })

product_embedding_df = pd.DataFrame(
    product_embedding_records
)

# 4. Create the product-level embedding matrix
product_embedding_matrix = np.stack(
    product_embedding_df["product_embedding"].to_numpy()
)

# 5. Validate
print("--- PRODUCT EMBEDDING SUMMARY ---")
print("Product embedding rows:", len(product_embedding_df))
print("Embedding matrix shape:", product_embedding_matrix.shape)
print(
    "Unique product IDs:",
    product_embedding_df["canonical_product_id"].nunique()
)
print(
    "Duplicate product IDs:",
    product_embedding_df["canonical_product_id"].duplicated().sum()
)

print("\n--- EMBEDDING COVERAGE ---")
print(
    product_embedding_df["embedding_image_count"]
    .value_counts()
    .sort_index()
)

print("\n--- NUMERICAL VALIDATION ---")
print("NaN values:", np.isnan(product_embedding_matrix).sum())
print("Infinite values:", np.isinf(product_embedding_matrix).sum())

embedding_norms = np.linalg.norm(
    product_embedding_matrix,
    axis=1
)

print("Minimum L2 norm:", embedding_norms.min())
print("Maximum L2 norm:", embedding_norms.max())

assert product_embedding_matrix.shape == (520, 512)
assert product_embedding_df["canonical_product_id"].is_unique
assert np.isfinite(product_embedding_matrix).all()
assert np.allclose(embedding_norms, 1.0, atol=1e-5)

print("\nPRODUCT-LEVEL EMBEDDING AGGREGATION COMPLETE.")


In [ ]:

# ============================================================
# PHASE 2 — FINAL PRODUCT-LEVEL CV FEATURE TABLE
# STEP 5
# IN-MEMORY ONLY — NO FILES SAVED
# ============================================================

# 1. Start with every canonical product
final_cv_features = products[
    ["canonical_product_id"]
].drop_duplicates().copy()

assert final_cv_features["canonical_product_id"].is_unique

# 2. Merge image-level aggregated features
final_cv_features = final_cv_features.merge(
    product_cv_features,
    on="canonical_product_id",
    how="left",
    validate="one_to_one"
)

# 3. Merge product-level embedding metadata
embedding_summary = product_embedding_df[
    ["canonical_product_id", "embedding_image_count"]
].copy()

final_cv_features = final_cv_features.merge(
    embedding_summary,
    on="canonical_product_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_embedding")
)

# 4. Fill observed-count fields only
count_columns = [
    "image_count_total",
    "image_count_downloaded",
    "image_count_pending",
    "image_count_with_local_path",
    "image_count_with_sha256",
    "image_count_with_dhash",
    "image_count_with_ocr",
    "image_count_with_packaging_score",
    "unique_sha256_count",
    "cross_product_duplicate_image_count",
    "cross_product_duplicate_hash_count",
    "distinct_other_products_sharing_image",
    "embedding_image_count"
]

for column in count_columns:
    if column in final_cv_features.columns:
        final_cv_features[column] = (
            final_cv_features[column].fillna(0).astype("int64")
        )

# 5. Assign explicit image evidence status
final_cv_features["image_evidence_status"] = "NO_IMAGE_RECORDS"

has_pending = final_cv_features["image_count_pending"] > 0
has_downloaded = final_cv_features["image_count_downloaded"] > 0

final_cv_features.loc[
    has_pending,
    "image_evidence_status"
] = "PENDING_IMAGE_ONLY"

final_cv_features.loc[
    has_downloaded & ~has_pending,
    "image_evidence_status"
] = "DOWNLOADED_IMAGE_EVIDENCE"

final_cv_features.loc[
    has_downloaded & has_pending,
    "image_evidence_status"
] = "PARTIALLY_DOWNLOADED_IMAGE_EVIDENCE"

# 6. Recalculate coverage ratios safely
final_cv_features["image_download_coverage"] = np.where(
    final_cv_features["image_count_total"] > 0,
    final_cv_features["image_count_downloaded"]
    / final_cv_features["image_count_total"],
    np.nan
)

final_cv_features["ocr_coverage"] = np.where(
    final_cv_features["image_count_downloaded"] > 0,
    final_cv_features["image_count_with_ocr"]
    / final_cv_features["image_count_downloaded"],
    np.nan
)

final_cv_features["embedding_coverage"] = np.where(
    final_cv_features["image_count_downloaded"] > 0,
    final_cv_features["embedding_image_count"]
    / final_cv_features["image_count_downloaded"],
    np.nan
)

# 7. Final structural validation
print("--- FINAL CV FEATURE TABLE ---")
print("Shape:", final_cv_features.shape)
print(
    "Unique product IDs:",
    final_cv_features["canonical_product_id"].nunique()
)
print(
    "Duplicate product IDs:",
    final_cv_features["canonical_product_id"].duplicated().sum()
)

print("\n--- IMAGE EVIDENCE STATUS ---")
print(
    final_cv_features["image_evidence_status"]
    .value_counts(dropna=False)
)

print("\n--- COVERAGE SUMMARY ---")
print(
    "Products with image records:",
    (final_cv_features["image_count_total"] > 0).sum()
)
print(
    "Products with downloaded images:",
    (final_cv_features["image_count_downloaded"] > 0).sum()
)
print(
    "Products with embeddings:",
    (final_cv_features["embedding_image_count"] > 0).sum()
)
print(
    "Products with cross-product duplicate evidence:",
    (
        final_cv_features[
            "cross_product_duplicate_image_count"
        ] > 0
    ).sum()
)

print("\n--- MISSING EVIDENCE CHECK ---")
print(
    "Products with no image records:",
    (
        final_cv_features["image_evidence_status"]
        == "NO_IMAGE_RECORDS"
    ).sum()
)

print(
    "Packaging score missing:",
    final_cv_features["packaging_score_mean"].isna().sum()
)

print("\n--- INTEGRITY CHECKS ---")

assert len(final_cv_features) == len(products)
assert final_cv_features["canonical_product_id"].is_unique
assert final_cv_features["canonical_product_id"].notna().all()
assert final_cv_features["image_count_total"].sum() == len(images)
assert (
    final_cv_features["image_count_downloaded"].sum()
    == (images["download_status"] == "DOWNLOADED").sum()
)
assert (
    final_cv_features["embedding_image_count"].sum()
    == len(embedding_metadata)
)

print("All structural and coverage checks passed.")
print("No files saved or modified.")


## 35. Final Certified Release Visual Intelligence — Dataset v1.2

> **v1.2 Additive Migration:** Evaluates image integrity, perceptual hash clusters, and cross-listing reuse on `data/marketplace_trust_dataset_v1.2/derived_evidence/image_integrity_features.parquet`.

In [ ]:
# Visual Intelligence Analysis for v1.2
from pathlib import Path
import pandas as pd

v12_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.2'
img_features = pd.read_parquet(v12_dir / 'derived_evidence' / 'image_integrity_features.parquet')
dup_clusters = img_features[img_features['is_duplicate_cluster'] == True]

print(f"Total Images Analyzed: {len(img_features):,}")
print(f"Usable Images: {(img_features['integrity_status'] == 'PASS').sum():,}")
print(f"Duplicate Image Clusters (dHash / SHA256 reuse): {len(dup_clusters):,} records")

## 36. Final Frozen Release Visual Intelligence — Dataset v1.2

> **v1.3 Additive Migration:** Evaluates image integrity, perceptual hash clusters, and cross-listing reuse on `data/marketplace_trust_dataset_v1.3/derived_evidence/image_integrity_features.parquet`.

In [ ]:
# Visual Intelligence Analysis for v1.2
from pathlib import Path
import pandas as pd

v13_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.3'
img_features = pd.read_parquet(v13_dir / 'derived_evidence' / 'image_integrity_features.parquet')
dup_clusters = img_features[img_features['is_duplicate_cluster'] == True]

print(f"Total Images Analyzed: {len(img_features):,}")
print(f"Usable Images: {(img_features['integrity_status'] == 'PASS').sum():,}")
print(f"Duplicate Image Clusters (dHash / SHA256 reuse): {len(dup_clusters):,} records")

## 36. Final Frozen Release Visual Intelligence — Dataset v1.2

> **v1.3 Additive Migration:** Evaluates image integrity, perceptual hash clusters, and cross-listing reuse on `data/marketplace_trust_dataset_v1.3/derived_evidence/image_integrity_features.parquet`.

In [ ]:
# Visual Intelligence Analysis for v1.2
from pathlib import Path
import pandas as pd

v13_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.3'
img_features = pd.read_parquet(v13_dir / 'derived_evidence' / 'image_integrity_features.parquet')
dup_clusters = img_features[img_features['is_duplicate_cluster'] == True]

print(f"Total Images Analyzed: {len(img_features):,}")
print(f"Usable Images: {(img_features['integrity_status'] == 'PASS').sum():,}")
print(f"Duplicate Image Clusters (dHash / SHA256 reuse): {len(dup_clusters):,} records")

## 37. Release v1.4 Visual Intelligence — Dataset v1.4

> **v1.4 Additive Migration:** Evaluates image integrity, perceptual hash clusters, and cross-listing reuse on `data/marketplace_trust_dataset_v1.4/derived_evidence/image_integrity_features.parquet`.

In [ ]:
# Visual Intelligence Analysis for v1.4
from pathlib import Path
import pandas as pd

v14_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.4'
img_features = pd.read_parquet(v14_dir / 'derived_evidence' / 'image_integrity_features.parquet')
dup_clusters = img_features[img_features['is_duplicate_cluster'] == True]

print(f"Total Images Analyzed: {len(img_features):,}")
print(f"Usable Images: {(img_features['integrity_status'] == 'PASS').sum():,}")
print(f"Duplicate Image Clusters (dHash / SHA256 reuse): {len(dup_clusters):,} records")

## 38. Release v1.5 Visual Integrity Intelligence

> **v1.5 Additive Section:** Audits visual assets, exact SHA-256 duplicates, and perceptual dHash collisions.

In [ ]:
# Visual Taxonomy and Integrity Audit
from pathlib import Path
import pandas as pd
from src.vision.vision_intelligence_v1_5 import VisionIntelligenceEngineV15

v15_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.5'
imgs_v15 = pd.read_parquet(v15_dir / 'evidence' / 'images.parquet')

vis_audit = VisionIntelligenceEngineV15.audit_visual_taxonomy(imgs_v15)
print("=== Visual Asset Taxonomy ===")
print(f"Total Usable Images: {vis_audit['total_usable_images']:,}")
print(f"Exact SHA-256 Duplicates: {vis_audit['exact_sha256_duplicates']}")
print(f"dHash Perceptual Collisions: {vis_audit['perceptual_dhash_collisions']}")
print(f"Resolution Breakdown: {vis_audit['resolution_breakdown']}")

## 39. Release v1.6 ResNet Deep Embedding Verification

> **v1.6 Additive Section:** Audits 512-dimensional deep visual embeddings extracted for all 5,228 verified marketplace listing images.

In [ ]:
# Deep Visual Embeddings Audit
from pathlib import Path
import pandas as pd

v16_dir = project_root / 'data' / 'marketplace_trust_dataset_v1.6'
vis_meta = pd.read_parquet(v16_dir / 'derived_evidence' / 'visual_embeddings_metadata.parquet')

print("=== Visual Embeddings Master Inventory ===")
print(f"Total Embedded Images: {len(vis_meta):,}")
print(f"Embedding Architecture: {vis_meta['embedding_model'].iloc[0]}")
print(f"Vector Dimensions: {vis_meta['embedding_dim'].iloc[0]}")
print(f"Mean L2-Norm: {vis_meta['l2_norm'].mean():.4f}")
display(vis_meta.groupby('marketplace').agg(
    image_count=('image_id', 'count'),
    mean_feature_val=('embedding_mean', 'mean')
))